1. 共用參數設定


In [ ]:
'''
Measure the elapsed time of every code cell, including cells that raise errors.
記錄每個 Code Cell 的執行時間，即使發生錯誤也顯示已耗費的時間。
'''
from time import perf_counter as _cell_perf_counter
from IPython import get_ipython as _cell_get_ipython

_CELL_TIMER_SHELL = _cell_get_ipython()
if _CELL_TIMER_SHELL is None:
    raise RuntimeError('請在 Jupyter Notebook 中執行本教材。')
_CELL_TIMER_STATE = {'started': _cell_perf_counter()}

def _cell_timer_begin(_info):
    _CELL_TIMER_STATE['started'] = _cell_perf_counter()

def _cell_timer_end(result):
    started = _CELL_TIMER_STATE.pop('started', None)
    if started is None:
        return
    minutes, seconds = divmod(_cell_perf_counter() - started, 60)
    elapsed = f'{int(minutes)} 分 {seconds:05.2f} 秒' if minutes else f'{seconds:.2f} 秒'
    status = '執行失敗' if result.error_before_exec or result.error_in_exec else '執行完成'
    print(f'⏱ 本格{status}，耗時：{elapsed}')

'''
Replace existing timers when the first cell is executed again.
重新執行第一格時，先取消舊計時器，避免重複顯示耗時。
'''
for _event_name, _callback in globals().get('_CELL_TIMER_CALLBACKS', ()):
    try:
        _CELL_TIMER_SHELL.events.unregister(_event_name, _callback)
    except ValueError:
        pass
_CELL_TIMER_CALLBACKS = (
    ('pre_run_cell', _cell_timer_begin),
    ('post_run_cell', _cell_timer_end),
)
for _event_name, _callback in _CELL_TIMER_CALLBACKS:
    _CELL_TIMER_SHELL.events.register(_event_name, _callback)

'''
Import standard-library tools for editable paths and local service settings.
匯入設定路徑與本機服務參數所需的 Python 內建工具。
'''
import os
from pathlib import Path
ROOT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.home() / 'ai_course'
ROOT = ROOT.expanduser().resolve()  # Ubuntu 可改成 Path('/home/your_user/ai_course')
SCRATCH_ROOT = Path('/tmp') if Path('/kaggle/working').is_dir() else ROOT / 'cache'
SCRATCH_ROOT = SCRATCH_ROOT.expanduser().resolve()  # 可改成其他儲存裝置或模型磁碟
ROOT.mkdir(parents=True, exist_ok=True)
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)

COMFY = ROOT / 'ComfyUI-MiniMaxH3'

'''
Store large downloaded GGUF weights in scratch storage.
設定 GGUF 模型權重的暫存磁碟位置，減少工作空間占用。
'''
STORE = SCRATCH_ROOT / 'minimax_h3_models'
PORT = 8193  # 本機可用 Port
H3_COMFY_REF = 'v0.45.0'

'''
Set the Hugging Face cache directory before any model downloads.
在下載模型之前指定 Hugging Face 快取目錄，避免大型檔案占滿工作空間。
'''
HF_CACHE_DIR = SCRATCH_ROOT / 'minimax_h3_hf'
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'

'''
Select the shared GGUF quantization. Larger formats may require CPU offload on T4.
選擇共用 GGUF 量化精度；越大的格式越可能需要 CPU Offload。
'''
QUANT = 'Q5_0'  # 'Q4_K', 'Q5_0', 'Q6_K', 'Q8_0'
DEFAULT_QUANT = QUANT

'''
Set GPU 0 memory budget in GiB; 14.5 is aggressive on a 15 GiB T4.
設定 GPU 0 的記憶體預算（GiB）；15 GiB T4 設為 14.5 剩餘緩衝很小，OOM 可調低。
'''
GPU_COUNT = 2  # Kaggle T4 ×2；其他 NVIDIA 主機可改成 1～4
MODEL_GPU_ID = 0  # 生成模型的主要運算 GPU（單張，並非分散式權重切割）
TEXT_GPU_ID = 1  # 文字編碼器 GPU，例如四張卡可指定為 3
VAE_GPU_ID = 1  # 影片／音訊 VAE 的 GPU；GPU 1 不足時可改回 0
GPU0_MODEL_GIB = 14.5  # 14.5、14.0、13.5
GPU1_MODEL_GIB = 14.5  # 14.5、14.0、13.5

'''
Select the text-encoder weight placement strategy.
設定文字編碼器權重管理：dynamic 分批調度較節省 GPU 記憶體；streaming 使用時整體載入；resident 盡量常駐 GPU 記憶體。
'''
TEXT_MODE = 'dynamic'  # 'dynamic', 'streaming', 'resident'
TEXT_GPU = f'cuda:{TEXT_GPU_ID}'  # 隨第一格 GPU ID 自動調整
NODE_CACHE_MODE = 'none'  # 影片生成不保留一般節點結果，優先節省 RAM

'''
Choose a post-generation policy: automatic, full cleanup, or GPU-preserving cache cleanup.
設定影片生成後的清理策略：自動、完整釋放，或只清節點快取保留 GPU 權重。
'''
VIDEO_RELEASE_MODE = 'full'  # 若保留服務，優先完整卸載模型與節點快取


'''
Keep idle GPU weights only when adequate host RAM remains for another video job.
只有系統 RAM 仍足夠支應下一支影片時，才繼續保留閒置 GPU 模型權重。
'''
VIDEO_KEEP_GPU_MIN_RAM_GIB = 12.0

'''
Stop ComfyUI after each saved video and restart before the next one.
影片完成後結束 ComfyUI，下次生成前自動啟動，避免 RAM 累積。
'''
VIDEO_RESTART_AFTER_GENERATION = True


'''
Select the optional remote FIFO API port; change only if the port conflicts.
指定選用 FIFO 遠端 API 的本機連接埠；只有發生 Port 衝突時才需要修改。
'''
'''
Keep generation and MP4 output at the H3-supported frame rate.
將影片生成與 MP4 輸出統一設定為 H3 支援的影格率。
'''
FPS = 24

REMOTE_PORT = 8765  # 例如 8765、8766，範圍 1024～65535
REMOTE_SERVICE_ID = '06_MiniMax_H3_影片生成_Kaggle'
REMOTE_ROOT = ROOT / 'remote_api_jobs' / REMOTE_SERVICE_ID

'''
Limit how many remote jobs wait for the single GPU worker.
設定 FIFO 最多等待的工作數量；超出時回傳 HTTP 429，client 會延遲重試。
'''
REMOTE_QUEUE_CAPACITY = 32  # 建議 12～32，依課堂人數調整
REMOTE_UPLOAD_CONCURRENCY = 2  # 建議 1～2
REMOTE_RESULT_TTL_HOURS = 24
REMOTE_RESULT_LIMIT_GIB = 8
REMOTE_MIN_FREE_GIB = 2

'''
Mark exceptionally long remote inference as stalled, without forcibly starting a second GPU job.
設定遠端推論超時警戒值；卡住時不強行啟動第二筆 GPU 推論。
'''
REMOTE_STALL_HOURS = float('inf')  # 不限制單次生成時長，也不將執行中的工作誤判為停滯
GGUF_REPO = 'robertbschuette/MiniMax-H3-GGUF'
R2V_REF_DIR = ROOT / 'minimax_h3_r2v_references'

'''
Validate user-editable local ports and remote queue limits.
確認教師設定的本機 Port 與遠端佇列限制有效，避免服務啟動後才失敗。
'''
assert 1024 <= REMOTE_PORT <= 65535, 'REMOTE_PORT 必須介於 1024～65535。'
assert 1 <= REMOTE_QUEUE_CAPACITY <= 1024, 'FIFO 佇列容量設定無效。'
assert 1 <= REMOTE_UPLOAD_CONCURRENCY <= 8, '上傳並發限制設定無效。'
assert REMOTE_RESULT_TTL_HOURS > 0 and REMOTE_RESULT_LIMIT_GIB > 0 and REMOTE_MIN_FREE_GIB >= 0
assert REMOTE_STALL_HOURS > 0
assert 1024 <= PORT <= 65535 and PORT != REMOTE_PORT
print('✓ 第 1 節完成：共用參數已設定，生成完成後關閉 ComfyUI，下次自動重啟。請繼續執行環境安裝。')

assert 1 <= GPU_COUNT <= 4, 'GPU_COUNT 必須介於 1～4。'
assert all(0 <= gpu < GPU_COUNT for gpu in (MODEL_GPU_ID, TEXT_GPU_ID)), 'GPU ID 超出 GPU_COUNT 的範圍。'
assert GPU_COUNT != 1 or MODEL_GPU_ID == TEXT_GPU_ID == VAE_GPU_ID, '單張 GPU 時，所有元件應使用相同的 GPU ID。'
assert 0 <= VAE_GPU_ID < GPU_COUNT, 'VAE_GPU_ID 超出 GPU_COUNT。'


2. 安裝環境與匯入套件


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import os
import subprocess
import sys
from pathlib import Path
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() >= GPU_COUNT, '可用 CUDA GPU 數量不足；請檢查 NVIDIA 驅動、PyTorch 與 GPU_COUNT。'
for i in sorted({MODEL_GPU_ID, TEXT_GPU_ID}):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}, {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GiB')

'''
Define the repositories needed for the local workflow.
定義本機 Workflow 需要的程式庫。
'''
repositories = {
    COMFY: 'https://github.com/Comfy-Org/ComfyUI.git',
    COMFY / 'custom_nodes/ComfyUI-GGUF': 'https://github.com/leejet/ComfyUI-GGUF.git',
    COMFY / 'custom_nodes/ComfyUI-ClipProj': 'https://github.com/nicolab28/ComfyUI-ClipProj.git',
    COMFY / 'custom_nodes/ComfyUI-MultiGPU': 'https://github.com/pollockjj/ComfyUI-MultiGPU.git',
}
old_commits = {}
for directory, url in repositories.items():
    if directory.exists() and not (directory / '.git').is_dir():
        raise RuntimeError(f'目錄不是 Git 倉庫：{directory}')
    if (directory / '.git').is_dir():
        old_commits[directory] = subprocess.check_output(['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True).strip()
    else:
        directory.parent.mkdir(parents=True, exist_ok=True)
        command = ['git', 'clone', '--depth', '1']

        '''
        Choose the release tag only for ComfyUI core.
        只有 ComfyUI 核心指定發行版標籤。
        '''
        if directory == COMFY:
            command += ['--branch', H3_COMFY_REF]
        subprocess.run(command + [url, str(directory)], check=True)

    '''
    The archived or patched GGUF loader may contain local compatibility changes.
    GGUF 載入器可能含有必要的本機相容性修正。
    '''
    if directory.name == 'ComfyUI-GGUF':
        dirty = subprocess.check_output(['git', '-C', str(directory), 'status', '--porcelain'], text=True).strip()
        if not dirty:
            subprocess.run(['git', '-C', str(directory), 'pull', '--ff-only', '-q'], check=True)
        else:
            print('GGUF 載入器已有本機修改，本次略過自動更新。')

'''
Retrieve the official ComfyUI release on repeat runs.
重跑時同樣取得官方 ComfyUI 發行版本。
'''
subprocess.run(['git', '-C', str(COMFY), 'fetch', '--depth', '1', 'origin', 'tag', H3_COMFY_REF], check=True)
subprocess.run(['git', '-C', str(COMFY), 'checkout', '--quiet', '--detach', H3_COMFY_REF], check=True)
H3_CORE_UPDATED = any(
    directory in old_commits and old_commits[directory] != subprocess.check_output(
        ['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True,
    ).strip()
    for directory in (COMFY, COMFY / 'custom_nodes/ComfyUI-GGUF')
)

'''
Update ClipProj to its current compatible implementation if this folder already existed.
若 ClipProj 資料夾原本就存在，同步更新到支援新版投影矩陣的程式。
'''
clipproj_dir = COMFY / 'custom_nodes/ComfyUI-ClipProj'
clipproj_before = subprocess.check_output(['git', '-C', str(clipproj_dir), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['git', '-C', str(clipproj_dir), 'fetch', '--quiet', '--depth', '1', 'origin', 'main'], check=True)

'''
Switch the local custom node to the retrieved revision for consistent code and weights.
切換到取得的版本，確保節點程式與投影矩陣相容。
'''
subprocess.run(['git', '-C', str(clipproj_dir), 'checkout', '--quiet', '--force', 'FETCH_HEAD'], check=True)
H3_CLIPPROJ_UPDATED = clipproj_before != subprocess.check_output(['git', '-C', str(clipproj_dir), 'rev-parse', 'HEAD'], text=True).strip()
clipproj_source = (clipproj_dir / 'clipproj_nodes.py').read_text(encoding='utf-8')

'''
Fail before loading models if an outdated fork lacks the required interfaces.
若外掛版本過舊且缺少必要介面，在載入模型前停止。
'''
assert 'def clip_types(' in clipproj_source and 'build_residual' in clipproj_source, 'ClipProj 缺少新版必要功能，請檢查安裝版本。'
constraint = ROOT / 'minimax_h3_torch_constraints.txt'
constraint.write_text(f'torch=={torch.__version__}\ntransformers>=4.50.3,<5.0\n', encoding='utf-8')

'''
Install ComfyUI dependencies and the GGUF backend.
安裝 ComfyUI 相依套件與 GGUF 後端。
'''
for requirements in [COMFY / 'requirements.txt', COMFY / 'custom_nodes/ComfyUI-GGUF/requirements.txt']:
    if requirements.is_file():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), '-r', str(requirements)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), 'huggingface_hub', 'hf_xet', 'requests', 'websocket-client', 'tqdm', 'ipywidgets>=8,<9'], check=True)
print('ComfyUI 與 MiniMax H3 節點安裝完成。')

'''
Import a small regex helper for the H3 GGUF architecture registration.
載入簡單的正規表示式工具，以註冊 H3 GGUF 架構。
'''
import re
gguf_loader_path = COMFY / 'custom_nodes/ComfyUI-GGUF/loader.py'
assert gguf_loader_path.is_file(), f'找不到 GGUF 載入器：{gguf_loader_path}'

'''
Read the exact installed GGUF implementation instead of assuming a fixed version.
讀取實際安裝的 GGUF 原始碼，不假設版本永遠固定。
'''
gguf_source = gguf_loader_path.read_text(encoding='utf-8')
arch_match = re.search(r'IMG_ARCH_LIST\s*=\s*\{[^}]*\}', gguf_source)
assert arch_match, '無法定位 IMG_ARCH_LIST；請檢查 ComfyUI-GGUF 版本。'

'''
Record whether the running ComfyUI process would require a reload.
記錄是否需要讓已啟動的 ComfyUI 重新載入節點。
'''
H3_RESTART_REQUIRED = H3_CLIPPROJ_UPDATED or H3_CORE_UPDATED or (COMFY / 'custom_nodes/ComfyUI-MultiGPU' not in old_commits)
if '"minimax_h3"' not in arch_match.group(0):
    previous_arch_list = arch_match.group(0)[:-1].rstrip()
    separator = '' if previous_arch_list.endswith(('{', ',')) else ','
    updated_arch_list = f'{previous_arch_list}{separator} "minimax_h3"}}'
    gguf_source = gguf_source[:arch_match.start()] + updated_arch_list + gguf_source[arch_match.end():]
    gguf_loader_path.write_text(gguf_source, encoding='utf-8')
    H3_RESTART_REQUIRED = True
    print('已加入 MiniMax-H3 GGUF 架構相容設定。')

print('✓ 第 2 節完成：ComfyUI、GGUF、ClipProj 與 WebSocket 工具已安裝／檢查。')


3. 下載量化模型（依共用路徑儲存）

In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import os
import shutil
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')

'''
Disable extra Xet chunk caching to conserve temporary disk space.
停用額外的 Xet 分塊快取，節省暫存空間。
'''
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'
from huggingface_hub import hf_hub_download

'''
Choose the generation mode from the four supported task strings. flf2v = first-and-last-frame-to-video; r2v = multi-reference-to-video (Ref2VA model). Available choices: t2v, i2v, flf2v, r2v.
從四種合法字串中選擇生成模式。 t2v = text-to-video (文字生成影片); i2v = image-to-video (首張圖片生成影片). t2v 為文生影片；i2v 為首張圖片生成影片。 flf2v 為首尾影格生成影片；r2v 為多張參考圖生成影片（需使用 Ref2VA 權重）。 可選值：t2v、i2v、flf2v、r2v。
'''
TASK = globals().get('TASK', 't2v')  # 't2v', 'i2v', 'l2v', 'flf2v', 'r2v'
assert TASK in ('t2v', 'i2v', 'l2v', 'flf2v', 'r2v'), f'不支援的 TASK：{TASK}'
MODEL_VARIANT = 'ref2va_pruned' if TASK == 'r2v' else 'fl2va_pruned'

'''
Use edited task parameters when the corresponding settings cell has run.
若已執行對應的參數 Cell，則優先採用使用者修改後的量化選項。
'''
task_settings = globals().get('TASK_CONFIGS', {}).get(TASK, {})
FL2VA_QUANT_SIZES_GIB = {'Q4_K': 10.64, 'Q5_0': 12.97, 'Q6_K': 15.45, 'Q8_0': 19.97}
REF2VA_QUANT_SIZES_GIB = {'Q4_K': 10.60, 'Q5_0': 12.94, 'Q6_K': 15.42, 'Q8_0': 19.94}

'''
Select the size table matching the actual diffusion model family.
依照真正要使用的模型種類選擇大小對照表。
'''
QUANT_SIZES_GIB = REF2VA_QUANT_SIZES_GIB if TASK == 'r2v' else FL2VA_QUANT_SIZES_GIB
assert QUANT in QUANT_SIZES_GIB, f'不支援 QUANT={QUANT!r}；請選擇 {list(QUANT_SIZES_GIB)}。'

'''
Build the actual published filename, never reusing FL2VA weights for R2V.
根據實際發布檔名選擇模型，R2V 絕不沿用 FL2VA 權重。
'''
DIT_NAME = f'minimax_h3_{MODEL_VARIANT}-{QUANT}.gguf'
models = [
    (GGUF_REPO, DIT_NAME, 'diffusion_models'),
    ('Comfy-Org/Krea-2', 'text_encoders/qwen3vl_4b_fp8_scaled.safetensors', 'text_encoders'),
    ('NicoLab28/ClipProj-MiniMax-H3', 'mmh3-4b-ClipProj-v3.1-mlp.safetensors', 'clip_projections'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_fp16.safetensors', 'vae'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors', 'vae'),
]
AUX_SIZES_GIB = {
    'qwen3vl_4b_fp8_scaled.safetensors': 4.88,
    'mmh3-4b-ClipProj-v3.1-mlp.safetensors': 0.47,
    'minimax_h3_video_vae_fp16.safetensors': 4.85,
    'minimax_h3_audio_vae_fp32.safetensors': 0.56,
}

'''
Keep the valid downloaded GGUF from either the current or historical folder.
可沿用目前或舊版資料夾中已完整下載的 GGUF 權重。
'''
GGUF_CACHE_PATHS = (STORE / 'diffusion_models' / DIT_NAME, STORE / 'unet_gguf' / DIT_NAME)
missing_gib = 0.0
for repo, filename, folder in models:
    size_gib = QUANT_SIZES_GIB[QUANT] if folder == 'diffusion_models' else AUX_SIZES_GIB[Path(filename).name]
    candidates = GGUF_CACHE_PATHS if folder == 'diffusion_models' else (STORE / folder / Path(filename).name,)
    complete = any(path.is_file() and 0.97 * size_gib <= path.stat().st_size / 1024**3 <= 1.03 * size_gib for path in candidates)

    '''
    Count only components that still need to be downloaded.
    只累計尚未完整下載的模型大小。
    '''
    if not complete:
        missing_gib += size_gib

'''
Read the free space of the scratch filesystem used for all model files.
讀取實際存放所有模型權重的暫存磁碟可用容量。
'''
tmp_free_gib = shutil.disk_usage('/tmp').free / 1024**3
assert tmp_free_gib >= missing_gib + 3, (
    f'/tmp 空間不足：仍需下載約 {missing_gib:.2f} GiB，'
    f'目前剩餘 {tmp_free_gib:.2f} GiB（另需預留 3 GiB）。'
)
print(f'選擇 {MODEL_VARIANT}/{QUANT}｜去噪權重約 {QUANT_SIZES_GIB[QUANT]:.2f} GiB｜'
      f'尚需下載約 {missing_gib:.2f} GiB｜/tmp 可用 {tmp_free_gib:.1f} GiB')

'''
Warn when high-bit quantizations will require greater GPU/CPU offload.
高精度量化需要較多 GPU／CPU 卸載時，事先提醒使用者。
'''
if QUANT in ('Q6_K', 'Q8_0'):
    print('注意：Q6／Q8 可能需要大量 CPU Offload，速度較慢且有記憶體不足風險。')

for repo, filename, folder in models:
    print('模型：', Path(filename).name, flush=True)
    source = next((path for path in GGUF_CACHE_PATHS
                   if folder == 'diffusion_models' and path.is_file()
                   and 0.97 * QUANT_SIZES_GIB[QUANT] <= path.stat().st_size / 1024**3 <= 1.03 * QUANT_SIZES_GIB[QUANT]), None)

    '''
    Download files with Hub-managed validation when no complete cache exists.
    沒有完整快取時，交由 Hugging Face Hub 管理下載及檔案驗證。
    '''
    if source is None:
        local_file = STORE / folder / Path(filename).name
        expected_gib = QUANT_SIZES_GIB[QUANT] if folder == 'diffusion_models' else AUX_SIZES_GIB[Path(filename).name]
        invalid_cache = local_file.is_file() and not (0.97 * expected_gib <= local_file.stat().st_size / 1024**3 <= 1.03 * expected_gib)

        '''
        Keep models under /tmp without copying gigabytes into Kaggle working.
        模型實體仍放在 /tmp，避免複製大型檔案到 Kaggle working。
        '''
        source = Path(hf_hub_download(repo_id=repo, filename=filename,
                                      local_dir=STORE / folder, force_download=invalid_cache))
    expected_gib = QUANT_SIZES_GIB[QUANT] if folder == 'diffusion_models' else AUX_SIZES_GIB[Path(filename).name]
    assert 0.97 * expected_gib <= source.stat().st_size / 1024**3 <= 1.03 * expected_gib, f'模型檔案大小異常：{source}'
    target = COMFY / 'models' / folder / source.name
    target.parent.mkdir(parents=True, exist_ok=True)

    '''
    Keep an already-correct symbolic link to avoid unnecessary service restarts.
    已指向正確權重的符號連結直接沿用，避免不必要的服務重新啟動。
    '''
    if target.is_symlink() and (not target.exists() or not target.samefile(source)):
        target.unlink()
    if not target.exists():
        target.symlink_to(source.resolve())
        H3_RESTART_REQUIRED = True
    assert target.samefile(source), f'模型檔案衝突：{target} 與預期來源 {source} 不同。請先檢查舊檔。'
    assert target.is_file(), f'模型連結失敗：{target}'
    print('ComfyUI 路徑：', target)

obsolete_link = COMFY / 'models' / 'unet_gguf' / DIT_NAME
if obsolete_link.is_symlink():
    obsolete_link.unlink()

'''
Confirm all requested model paths can be opened.
確認所有所需模型檔案都可正常存取。
'''
assert all((COMFY / 'models' / folder / Path(filename).name).is_file() for _, filename, folder in models)
print(f'H3 {MODEL_VARIANT}/{QUANT}、4B ClipProj、Video VAE、Audio VAE 已就緒。')
H3_MODEL_CHOICE = (MODEL_VARIANT, QUANT)

'''
Remind users to revalidate the running loader after changing quantizations.
提醒切換量化格式後，先更新目前 ComfyUI 的模型選單。
'''
print('切換量化格式後：請重新執行第 4 節，再執行第 11 節生成。')
print(f'✓ 第 3 節完成：{MODEL_VARIANT}/{QUANT} 模型與相關權重已備妥（/tmp）。')


4. GPU 記憶體與 CPU Offload 設定


In [ ]:
'''
Require the teacher settings cell before applying ComfyUI GPU configuration.
套用 ComfyUI GPU 設定前，確認已執行教師共用設定 Cell。
'''
assert all(name in globals() for name in ('GPU0_MODEL_GIB', 'GPU1_MODEL_GIB', 'TEXT_MODE', 'TEXT_GPU', 'NODE_CACHE_MODE')), '請先執行「教師共用設定」Cell。'
gpu0_capacity = torch.cuda.get_device_properties(MODEL_GPU_ID).total_memory / 1024**3
gpu1_capacity = torch.cuda.get_device_properties(TEXT_GPU_ID).total_memory / 1024**3

'''
Reject values that exceed the available GPU memory.
避免設定超過 GPU 實際容量的數值。
'''
assert 0 < GPU0_MODEL_GIB <= gpu0_capacity, f'GPU0_MODEL_GIB 必須介於 0 和 {gpu0_capacity:.2f} GiB 之間。'
assert 0 < GPU1_MODEL_GIB <= gpu1_capacity, f'GPU1_MODEL_GIB 必須介於 0 和 {gpu1_capacity:.2f} GiB 之間。'
assert TEXT_MODE in ('dynamic', 'streaming', 'resident'), 'TEXT_MODE 設定無效。'

'''
Reserve the difference for GPU 0 so ComfyUI targets the chosen overall budget.
將 GPU 0 的容量差額保留，讓 ComfyUI 以指定的整體預算運作。
'''
GPU0_RESERVE_GIB = round(gpu0_capacity - GPU0_MODEL_GIB, 3)

assert NODE_CACHE_MODE in ('classic', 'none')
H3_DESIRED_CONFIG = (GPU0_RESERVE_GIB, NODE_CACHE_MODE, MODEL_GPU_ID, TEXT_GPU_ID, VAE_GPU_ID)

'''
Clarify that per-model residency is not a hard limit.
明確提醒：這不是單一模型的精確權重常駐限制。
'''
print('生成 GPU：ComfyUI Dynamic VRAM / CPU Offload（權重、運算暫存及其它元件共用預算）。')
print(f'文字編碼 GPU：ClipProj {TEXT_MODE}，{GPU1_MODEL_GIB:.2f} GiB 為監控警示值，並非硬上限。')
print(f'✓ 第 4 節完成：GPU 0 管理預算約 {GPU0_MODEL_GIB:.2f} GiB；保留 {GPU0_RESERVE_GIB:.2f} GB。')


5. 啟動本機 ComfyUI 與相依性檢查


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import requests
import time
LOG = ROOT / 'minimax_h3_comfyui.log'
BASE = f'http://127.0.0.1:{PORT}'

'''
Read server readiness without requiring a public tunnel.
不開放公開連線即可檢查伺服器狀態。
'''
def ready():
    '''
    Handle an unavailable local service safely.
    安全處理尚未啟動的服務。
    '''
    try:
        return requests.get(f'{BASE}/system_stats', timeout=3).ok

    except requests.RequestException:
        return False

'''
Read the ComfyUI loader's actual model dropdown values.
讀取 ComfyUI GGUF Loader 真正提供的模型選項。
'''
def listed_ggufs():
    '''
    Retrieve the registered GGUF model names from the local service.
    從本機服務取得已註冊的 GGUF 模型檔名。
    '''
    payload = requests.get(f'{BASE}/object_info/UnetLoaderGGUF', timeout=30).json()
    info = payload.get('UnetLoaderGGUF', {})
    spec = info.get('input', {}).get('required', {}).get('unet_name')
    return spec[0] if isinstance(spec, list) and spec and isinstance(spec[0], list) else []

need_restart = False
if ready():
    need_restart = (
        globals().get('H3_RESTART_REQUIRED', False)
        or DIT_NAME not in listed_ggufs()
        or globals().get('H3_ACTIVE_MODEL') != (MODEL_VARIANT, QUANT)
        or globals().get('H3_ACTIVE_CONFIG') != H3_DESIRED_CONFIG
    )

'''
Never terminate services launched by another notebook or process.
不任意終止其它 Notebook 或行程啟動的 ComfyUI。
'''
if need_restart:
    '''
    Only restart a service for which this kernel holds the process handle.
    只有目前 Kernel 持有行程控制權時才能自動重新啟動。
    '''
    assert 'H3_PROC' in globals() and H3_PROC.poll() is None, (
        '服務需要更新記憶體設定，但不是本 Notebook 可管理的行程。'
        f'請先停止連接埠 {PORT} 的舊服務，再執行本 Cell。'
    )
    H3_PROC.terminate()
    try:
        H3_PROC.wait(timeout=20)

    except subprocess.TimeoutExpired:
        '''
        Handle a service that does not respond to graceful shutdown.
        處理無法正常關閉的服務。
        '''
        H3_PROC.kill()
        H3_PROC.wait(timeout=10)
    print('重新啟動 ComfyUI：模型索引、節點版本或 GPU 0 記憶體預算有變更。')

if not ready():
    command = [
        sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', str(PORT),
        '--default-device', str(MODEL_GPU_ID),
        '--enable-dynamic-vram', '--reserve-vram', str(GPU0_RESERVE_GIB),
        '--cache-classic' if NODE_CACHE_MODE == 'classic' else '--cache-none',
        '--disable-pinned-memory',
    ]

    '''
    Keep the local service log available for debugging without blocking Kaggle.
    保留服務 Log 方便除錯，且不讓 Notebook 卡住。
    '''
    with LOG.open('w', encoding='utf-8') as stream:
        H3_PROC = subprocess.Popen(command, cwd=COMFY, stdout=stream, stderr=subprocess.STDOUT)
    for attempt in range(180):
        '''
        Report the original service error rather than waiting indefinitely.
        發生啟動錯誤時直接輸出 Log，避免無限等待。
        '''
        if H3_PROC.poll() is not None:
            raise RuntimeError(LOG.read_text(errors='replace')[-4000:])
        if ready():
            break
        time.sleep(2)
    assert ready(), LOG.read_text(errors='replace')[-4000:]

H3_ACTIVE_CONFIG = H3_DESIRED_CONFIG

'''
Get the exact input schemas registered by the running ComfyUI process.
從目前執行中的 ComfyUI 讀取真正註冊的節點輸入規格。
'''
NODE_INFO = requests.get(f'{BASE}/object_info', timeout=90).json()
REQUIRED_NODES = (
    'ClipProjLoader', 'UnetLoaderGGUF', 'MiniMaxH3SigmaShift',
    'VAELoaderMultiGPU', 'MiniMaxH3ImageToVideo', 'BasicGuider', 'KSamplerSelect',
    'BasicScheduler', 'RandomNoise', 'SamplerCustomAdvanced', 'VAEDecode',
    'VAEDecodeAudio', 'CreateVideo', 'SaveVideo', 'LoadImage',
    'MiniMaxH3ReferenceToVideo',
)
missing = [name for name in REQUIRED_NODES if name not in NODE_INFO]

'''
Show a clear explanation for a failed or incomplete ComfyUI installation.
若 ComfyUI 安裝不完整，直接指出缺少的節點。
'''
assert not missing, f'ComfyUI 缺少節點：{missing}。請檢查 {LOG} 並重新執行安裝與啟動。'

'''
Confirm the selected auxiliary VAE device is accepted by the installed MultiGPU loader.
確認本機 MultiGPU VAE Loader 實際支援所選的輔助 GPU。
'''
vae_devices = choices('VAELoaderMultiGPU', 'device')
if vae_devices is not None:
    assert f'cuda:{VAE_GPU_ID}' in vae_devices, f'VAE Loader 不支援 cuda:{VAE_GPU_ID}。'

'''
Read a dropdown's allowed values from ComfyUI's live schema.
從 ComfyUI 正在使用的規格讀取下拉選單允許值。
'''
def choices(node_name, input_name):
    '''
    Support both required and optional node inputs.
    同時支援必填與選填的節點輸入。
    '''
    groups = NODE_INFO[node_name].get('input', {})
    for group in ('required', 'optional'):
        spec = groups.get(group, {}).get(input_name)

        '''
        Return None for numeric/string inputs that are not dropdown lists.
        如果不是下拉式選單，就不視為有限選項。
        '''
        if spec is not None:
            if isinstance(spec, (list, tuple)) and spec:
                if isinstance(spec[0], list):
                    return spec[0]
                if len(spec) > 1 and isinstance(spec[1], dict):
                    for key in ('choices', 'options'):
                        if isinstance(spec[1].get(key), list):
                            return spec[1][key]
            return None
    raise RuntimeError(f'節點 {node_name} 沒有預期的輸入 {input_name}；可能是節點版本不相容.')

'''
Validate dropdown values without crashing if a node exposes only a scalar input.
驗證下拉選項；如果節點只提供一般輸入型別，就不要因此發生例外。
'''
def check_choice(node_name, input_name, value, fallback=None):
    '''
    Read the node's current explicit choice list if present.
    有明確選項清單時優先讀取目前節點的清單。
    '''
    allowed = choices(node_name, input_name)
    if allowed is None and fallback and fallback[0] in NODE_INFO:
        allowed = choices(*fallback)
    if allowed is None:
        print(f'預檢提示：{node_name}.{input_name} 沒有選項清單，將交由 ComfyUI 驗證。')
    else:
        assert value in allowed, f'{node_name}.{input_name} 不支援 {value!r}；可用選項：{allowed}'

assert DIT_NAME in (choices('UnetLoaderGGUF', 'unet_name') or []), f'GGUF Loader 沒有辨識 {DIT_NAME}。'
assert 'qwen3vl_4b_fp8_scaled.safetensors' in (choices('ClipProjLoader', 'clip_name') or []), 'ClipProj Loader 沒有辨識 4B Encoder。'

'''
Verify that the correct projection matrix is registered, not merely downloaded.
確認正確的投影矩陣已註冊，而不只是檔案下載成功。
'''
assert 'mmh3-4b-ClipProj-v3.1-mlp.safetensors' in (choices('ClipProjLoader', 'projection') or []), 'ClipProj Loader 沒有辨識 v3.1 投影矩陣。'
assert 'auto' in (choices('ClipProjLoader', 'type') or []), 'ClipProj Loader 不支援 type=auto，請更新節點。'
assert 'krea2' in (choices('ClipProjLoader', 'type') or []), '目前 ComfyUI 尚未提供 krea2／Qwen3-VL 4B 編碼器載入支援，請更新 ComfyUI。'

'''
Verify the supported video and audio VAE filenames.
確認影片與音訊 VAE 都出現在實際模型選單中。
'''
for vae_name in ('minimax_h3_video_vae_fp16.safetensors', 'minimax_h3_audio_vae_fp32.safetensors'):
    assert vae_name in (choices('VAELoader', 'vae_name') or []), f'ComfyUI 沒有辨識 VAE：{vae_name}'

print('依賴檢查通過：FL2VA、Ref2VA、GGUF、4B Encoder、ClipProj v3.1、Video/Audio VAE。')
H3_ACTIVE_QUANT = QUANT

'''
Remember the model family as well as its quantization.
同時記錄模型種類及量化格式，避免混用 FL2VA／Ref2VA。
'''
H3_ACTIVE_MODEL = (MODEL_VARIANT, QUANT)
H3_RESTART_REQUIRED = False
print(f'本機 MiniMax H3 {MODEL_VARIANT}/{QUANT} 已就緒（GGUF 模型選單驗證通過）。')

'''
Confirm that all service and model checks finished without an exception.
所有服務及模型檢查通過後，明確顯示本格完成。
'''
print(f'✓ 第 5 節完成：ComfyUI 服務與模型預檢完成；GPU 0 保留 {GPU0_RESERVE_GIB:.2f} GB。')

'''
Use a single managed ComfyUI lifecycle for notebook and API requests.
Notebook 手動生成與遠端 API 共用同一套 ComfyUI 行程生命週期。
'''
def h3_ensure_service():
    '''
    Start the owned ComfyUI process and verify the required model loaders.
    啟動本 Notebook 管理的 ComfyUI，並確認必要模型節點可使用。
    '''
    global H3_PROC, NODE_INFO

    '''
    Reject model changes that require redoing the setup cells.
    模型設定已改變時，要求重新執行安裝與模型設定 Cell。
    '''
    if H3_ACTIVE_CONFIG != H3_DESIRED_CONFIG or globals().get('H3_RESTART_REQUIRED', False):
        raise RuntimeError('模型或 GPU 設定已變更，請重新執行模型下載與 ComfyUI 啟動 Cell。')

    '''
    Reuse a healthy ComfyUI process without duplicating models.
    ComfyUI 已健康啟動時直接重用，避免重複載入模型。
    '''
    process = globals().get('H3_PROC')
    if process is not None and process.poll() is None and ready():
        return process

    '''
    Avoid controlling a ComfyUI server that belongs to another notebook.
    防止控制到其他 Notebook 的 ComfyUI 服務。
    '''
    if ready():
        raise RuntimeError(f'Port {PORT} 被其他 ComfyUI 服務占用，請先停止舊服務。')

    '''
    Stop an unresponsive but owned previous process without making duplicates.
    舊行程失去回應時，先安全停止再重新啟動。
    '''
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired as error:
            raise RuntimeError('ComfyUI 舊行程無法正常停止，已避免重複啟動。') from error

    '''
    Reuse the exact original ComfyUI flags and device configuration.
    延用原本模型啟動的 ComfyUI 參數與 GPU 分配。
    '''
    command = [
        sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', str(PORT),
        '--default-device', str(MODEL_GPU_ID),
        '--enable-dynamic-vram', '--reserve-vram', str(GPU0_RESERVE_GIB),
        '--cache-classic' if NODE_CACHE_MODE == 'classic' else '--cache-none',
        '--disable-pinned-memory',
    ]

    '''
    Start a fresh process that cannot inherit the old model's CPU RAM.
    啟動全新行程，避免沿用上一支影片的 CPU RAM 模型快取。
    '''
    with LOG.open('w', encoding='utf-8') as stream:
        H3_PROC = subprocess.Popen(command, cwd=COMFY, stdout=stream, stderr=subprocess.STDOUT)

    '''
    Verify the complete node registry and selected model before inference.
    在推論前驗證必要節點及所選模型皆已註冊。
    '''
    for _ in range(180):
        if H3_PROC.poll() is not None:
            raise RuntimeError('ComfyUI 啟動失敗：\n' + LOG.read_text(errors='replace')[-5000:])
        if ready():
            try:
                info = requests.get(f'{BASE}/object_info', timeout=30).json()
                missing = [name for name in REQUIRED_NODES if name not in info]
                selected = info.get('UnetLoaderGGUF', {}).get('input', {}).get('required', {}).get('unet_name', [[]])[0]
                if not missing and DIT_NAME in selected:
                    NODE_INFO = info
                    print(f'ComfyUI 已重新啟動 (PID {H3_PROC.pid})；模型節點檢查通過。')
                    return H3_PROC
            except (requests.RequestException, ValueError, KeyError, IndexError, TypeError):
                pass
        time.sleep(2)
    raise RuntimeError('ComfyUI 啟動或模型節點註冊逾時：\n' + LOG.read_text(errors='replace')[-5000:])


'''
Stop only the owned and idle ComfyUI process after a video job finishes.
影片工作完成後，只停止本 Notebook 管理且佇列空閒的 ComfyUI 行程。
'''
def h3_stop_service():
    '''
    Avoid stopping any process other than this notebook's child process.
    只停止目前 Notebook 自己管理的子行程。
    '''
    process = globals().get('H3_PROC')
    if process is None or process.poll() is not None:
        return True

    '''
    Check the ComfyUI queue to prevent terminating unfinished jobs.
    確認 ComfyUI 沒有正在執行或等待中的工作。
    '''
    if ready():
        try:
            response = requests.get(BASE + '/queue', timeout=8)
            response.raise_for_status()
            queue = response.json()
            if queue.get('queue_running') or queue.get('queue_pending'):
                print('ComfyUI 尚有工作，本次不關閉服務。')
                return False
        except (requests.RequestException, ValueError) as error:
            print(f'無法確認 ComfyUI 佇列空閒，本次不關閉服務：{error}')
            return False

    '''
    Let the operating system reclaim model RAM after process exit.
    結束 ComfyUI 行程後，由作業系統回收模型占用的 RAM。
    '''
    try:
        process.terminate()
        process.wait(timeout=30)
        print('ComfyUI 已停止；下一筆生成前將自動重新啟動。')
        return True
    except subprocess.TimeoutExpired:
        print('ComfyUI 停止逾時；為避免誤殺生成工作，不再強制終止。')
        return False
    except OSError as error:
        print(f'ComfyUI 停止失敗：{error}')
        return False


6. T2V：Prompt 與參數設定


In [ ]:
'''
Describe the text-to-video scene and synchronized sound.
設定文生影片的場景與同步音效描述。
'''
T2V_PROMPT = '''
A cinematic close-up of a red panda in a misty forest. The red panda slowly turns its head.
Natural fur details, gentle camera movement, soft morning light. Sound: birds chirping and light wind.
'''.strip()
T2V_WIDTH, T2V_HEIGHT = 384, 224
T2V_SECONDS = 10

'''
Choose sampling iterations for the original model.
設定原始模型的採樣步數。
'''
T2V_STEPS = 20
T2V_AUDIO_SHIFT = 6.0
T2V_SEED = 42

'''
Set the T2V output filename without overwriting earlier runs.
設定文生影片輸出檔名；若同名會自動編號。
'''
T2V_OUTPUT = 'minimax_h3_t2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})
TASK_CONFIGS['t2v'] = dict(prompt=T2V_PROMPT, width=T2V_WIDTH, height=T2V_HEIGHT,
                           fps=FPS, seconds=T2V_SECONDS, steps=T2V_STEPS, seed=T2V_SEED,
                           audio_shift=T2V_AUDIO_SHIFT, quant=QUANT,
                           output_name=T2V_OUTPUT, first_image='', last_image='')

'''
Provide explicit feedback after configuring the task.
完成任務設定後顯示明確提示。
'''
TASK = 't2v'

print(f'✓ 第 6 節完成：T2V | {T2V_WIDTH}×{T2V_HEIGHT} | {T2V_SECONDS} 秒 | {T2V_STEPS} steps | {QUANT} | Audio Shift {T2V_AUDIO_SHIFT} | {T2V_OUTPUT}')


7. I2V：Prompt、起始圖片與參數設定


In [ ]:
'''
Describe how the uploaded first frame should animate.
描述上傳的第一張圖片要如何轉化為動態影片。
'''
I2V_PROMPT = '''
Animate the subject in the reference image with subtle natural motion and smooth cinematic camera movement.
Keep the person's or object's identity, clothing and environment consistent. Include realistic ambient sound.
'''.strip()
I2V_FIRST_IMAGE = str(ROOT / 'first_frame.png')
I2V_WIDTH, I2V_HEIGHT = 384, 224

'''
Configure output duration for image-to-video.
設定圖生影片的目標秒數。
'''
I2V_SECONDS = 10
I2V_STEPS = 20
I2V_AUDIO_SHIFT = 6.0

'''
Set this task’s seed; use -1 to randomize each generation.
設定本次任務的種子；-1 表示每次重新抽取。量化設定沿用第 1 節的 QUANT。
'''
I2V_SEED = 42
I2V_OUTPUT = 'minimax_h3_i2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Register the first-frame conditioning input and task parameters.
登記起始影格及圖生影片參數。
'''
TASK_CONFIGS['i2v'] = dict(prompt=I2V_PROMPT, width=I2V_WIDTH, height=I2V_HEIGHT,
                           fps=FPS, seconds=I2V_SECONDS, steps=I2V_STEPS, seed=I2V_SEED,
                           audio_shift=I2V_AUDIO_SHIFT, quant=QUANT,
                           output_name=I2V_OUTPUT, first_image=I2V_FIRST_IMAGE, last_image='')

'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 'i2v'

print(f'✓ 第 7 節完成：I2V | {I2V_WIDTH}×{I2V_HEIGHT} | {I2V_SECONDS} 秒 | {QUANT} | Audio Shift {I2V_AUDIO_SHIFT} | 圖片：{I2V_FIRST_IMAGE}')


8. L2VA：最後影格生成影片


In [ ]:
'''
Describe how the uploaded last frame should animate.
描述上傳的最後一張圖片要如何轉化為動態影片。
'''
L2V_PROMPT = '''
Animate the subject in the reference image with subtle natural motion and smooth cinematic camera movement.
Keep the person's or object's identity, clothing and environment consistent. Include realistic ambient sound.
'''.strip()
L2V_LAST_IMAGE = str(ROOT / 'last_frame.png')
L2V_WIDTH, L2V_HEIGHT = 384, 224

'''
Configure output duration for image-to-video.
設定最後影格生成影片的目標秒數。
'''
L2V_SECONDS = 10
L2V_STEPS = 20
L2V_AUDIO_SHIFT = 6.0

'''
Set this task’s seed; use -1 to randomize each generation.
設定本次任務的種子；-1 表示每次重新抽取。量化設定沿用第 1 節的 QUANT。
'''
L2V_SEED = 42
L2V_OUTPUT = 'minimax_h3_l2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Register the last-frame conditioning input and task parameters.
登記最後影格及圖生影片參數。
'''
TASK_CONFIGS['l2v'] = dict(prompt=L2V_PROMPT, width=L2V_WIDTH, height=L2V_HEIGHT,
                           fps=FPS, seconds=L2V_SECONDS, steps=L2V_STEPS, seed=L2V_SEED,
                           audio_shift=L2V_AUDIO_SHIFT, quant=QUANT,
                           output_name=L2V_OUTPUT, first_image='', last_image=L2V_LAST_IMAGE)

'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 'l2v'

print(f'✓ 第 8 節完成：L2V | {L2V_WIDTH}×{L2V_HEIGHT} | {L2V_SECONDS} 秒 | {QUANT} | Audio Shift {L2V_AUDIO_SHIFT} | 圖片：{L2V_LAST_IMAGE}')


9. FLF2V：Prompt、首尾圖片與參數設定


In [ ]:
'''
Describe a video that connects its first and last reference frames.
描述從第一張圖片過渡到最後一張圖片的影片內容。
'''
FLF2V_PROMPT = '''
Create a smooth cinematic transition from the first reference frame to the last reference frame.
Maintain consistent subject details and lighting, with natural intermediate motion and synchronized ambient sound.
'''.strip()
FLF2V_FIRST_IMAGE = str(ROOT / 'first_frame.png')
FLF2V_LAST_IMAGE = str(ROOT / 'last_frame.png')

'''
Set output dimensions for first-and-last-frame generation.
設定首尾影格模式的輸出解析度。
'''
FLF2V_WIDTH, FLF2V_HEIGHT = 384, 224
FLF2V_SECONDS = 10
FLF2V_STEPS = 20

'''
Set the audio sigma shift for this generation mode (ComfyUI default: 3.0).
設定這個生成模式的音訊 Sigma Shift（ComfyUI 預設：3.0）。
'''
FLF2V_AUDIO_SHIFT = 6.0
FLF2V_SEED = 42
FLF2V_OUTPUT = 'minimax_h3_flf2v.mp4'

'''
Keep previously configured modes unchanged.
保留先前其它生成模式的設定。
'''
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})
TASK_CONFIGS['flf2v'] = dict(prompt=FLF2V_PROMPT, width=FLF2V_WIDTH, height=FLF2V_HEIGHT,
                             fps=FPS, seconds=FLF2V_SECONDS, steps=FLF2V_STEPS, seed=FLF2V_SEED,
                             audio_shift=FLF2V_AUDIO_SHIFT, quant=QUANT,
                             output_name=FLF2V_OUTPUT, first_image=FLF2V_FIRST_IMAGE,
                             last_image=FLF2V_LAST_IMAGE)

'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 'flf2v'

print(f'✓ 第 9 節完成：FLF2V | {FLF2V_WIDTH}×{FLF2V_HEIGHT} | {FLF2V_SECONDS} 秒 | {QUANT} | Audio Shift {FLF2V_AUDIO_SHIFT} | 參考圖 2 張')


10. R2V：Prompt、圖片／影片／音訊參考


In [ ]:
'''
Describe how multiple reference images define the subject and environment.
使用多張參考圖片指定人物外觀、服裝及環境。
'''
R2V_PROMPT = '''
<Picture 1> defines the subject's face, hairstyle, and identity.
Additional pictures, when provided, help specify the outfit and environment.
Create a cinematic video of the subject walking naturally through the scene.
Preserve identity, clothing, and scene details from the reference pictures.
Natural motion, soft daylight, realistic ambient sounds.
'''.strip()
R2V_REF_IMAGES = []
R2V_REF_VIDEOS = []  # Example: [str(ROOT / 'clip.mp4')]

'''
Optional standalone reference audio files (up to three).
Ref2VA 可選的獨立音訊路徑（最多三段，依 <Audio N> 編號排序）。
'''
R2V_REF_AUDIOS = []  # Example: [str(ROOT / 'voice.wav')]
R2V_VIDEO_AUDIO = False  # True, False
R2V_REF_IMAGE_SIZE = 'match'

'''
Select the R2V-specific ClipProj mode: resident, dynamic, or streaming. resident = prioritize vision-encoder stability; dynamic = lower residency; streaming = page weights. Available choices: resident, dynamic, streaming.
R2V 專用 ClipProj 模式可選 resident、dynamic、streaming。 resident 優先保持影像編碼器常駐；dynamic 動態卸載；streaming 使用分段載入。 可選值：resident、dynamic、streaming。
'''
R2V_TEXT_MODE = TEXT_MODE
R2V_WIDTH, R2V_HEIGHT = 384, 224
R2V_SECONDS = 10

'''
Set the diffusion sampling steps for the normal Ref2VA model.
設定正常版 Ref2VA 模型的採樣步數。
'''
R2V_STEPS = 20
R2V_AUDIO_SHIFT = 3.0

'''
Set this task’s seed; use -1 to randomize each generation.
設定此任務的種子；-1 代表每次生成都重新抽取。
'''
R2V_SEED = 42
R2V_OUTPUT = 'minimax_h3_r2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Save the reference media and settings without downloading extra models yet.
儲存參考圖與參數，不在此 Cell 下載額外權重。
'''
TASK_CONFIGS['r2v'] = dict(prompt=R2V_PROMPT, width=R2V_WIDTH, height=R2V_HEIGHT,
                            fps=FPS, seconds=R2V_SECONDS, steps=R2V_STEPS, seed=R2V_SEED,
                            audio_shift=R2V_AUDIO_SHIFT, quant=QUANT,
                            output_name=R2V_OUTPUT, first_image='', last_image='',
                            ref_images=R2V_REF_IMAGES, ref_image_size=R2V_REF_IMAGE_SIZE,
                            ref_videos=R2V_REF_VIDEOS, ref_audios=R2V_REF_AUDIOS,
                            video_audio=R2V_VIDEO_AUDIO, text_mode=R2V_TEXT_MODE)

'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 'r2v'

print(f'✓ 第 10 節完成：R2V | {R2V_WIDTH}×{R2V_HEIGHT} | {R2V_SECONDS} 秒 | '
      f'{QUANT} | {len(R2V_REF_IMAGES)} 張圖片；若尚未下載 R2V 權重，請重新執行第 2、4 節。')

import ipywidgets as widgets
from pathlib import Path
from PIL import Image, ImageOps
from io import BytesIO
from uuid import uuid4
from html import escape
from IPython.display import display, clear_output
import shutil
_REF_ROOT = R2V_REF_DIR
_REF_ROOT.mkdir(parents=True, exist_ok=True)
_REF_MAX = 9
_REF_FILES = sorted(p for p in _REF_ROOT.iterdir() if p.is_file() and p.suffix.lower() in ('.png','.jpg','.jpeg','.webp'))
if len(_REF_FILES) > _REF_MAX or any(p.stem != f'{i:03d}' for i,p in enumerate(_REF_FILES,1)):
    raise RuntimeError('參考圖編號不連續；請清理舊資料夾再執行上傳 Cell。')

'''
Synchronize the numbered paths with the generation configuration.
將最新圖片清單同步到生成工作所使用的設定。
'''
def _ref_sync(paths):
    global R2V_REF_IMAGES
    R2V_REF_IMAGES = [str(path) for path in paths]
    TASK_CONFIGS['r2v']['ref_images'] = list(R2V_REF_IMAGES)

'''
Replace all reference images only after the new upload is fully validated.
新圖片全部通過驗證後才取代上一批；選得更少時也會清除多餘舊檔。
'''
def _ref_update(change):
    items = change['new']
    if not items:
        return
    if isinstance(items, dict):
        items = [dict(name=name, **item) for name, item in items.items()]
    stage = _REF_ROOT.with_name(_REF_ROOT.name + '_stage_' + uuid4().hex)
    stage.mkdir(parents=True)
    backup = _REF_ROOT.with_name(_REF_ROOT.name + '_backup_' + uuid4().hex)
    try:
        if not 1 <= len(items) <= _REF_MAX:
            raise ValueError(f'一次請選擇 1～{_REF_MAX} 張圖片。')
        for index, item in enumerate(items, 1):
            filename = Path(item['name']).name
            suffix = Path(filename).suffix.lower()
            if suffix not in ('.png','.jpg','.jpeg','.webp'):
                raise ValueError(f'不支援的圖片格式：{filename}')
            blob = bytes(item['content'])
            if len(blob) > 20 * 1024 * 1024:
                raise ValueError(f'單張圖片超過 20 MiB：{filename}')
            with Image.open(BytesIO(blob)) as photo:
                photo.verify()
            (stage / f'{index:03d}{suffix}').write_bytes(blob)
        _REF_ROOT.rename(backup)
        try:
            stage.rename(_REF_ROOT)
        except Exception:
            backup.rename(_REF_ROOT)
            raise
        shutil.rmtree(backup)
        current = sorted(_REF_ROOT.iterdir())
        _ref_sync(current)
        print('最新參考圖：', ', '.join(f'{p.name} = <Picture {i}>' for i,p in enumerate(current,1)))
    except Exception as error:
        shutil.rmtree(stage, ignore_errors=True)
        print('上傳未生效，保留上一批參考圖：', error)
    finally:
        _REF_PANEL.children = (_ref_new(),)

'''
Reset the browser picker so selecting the same filenames works again.
每次上傳後重建選檔元件，確保可以重新選相同檔名。
'''
def _ref_new():
    picker = widgets.FileUpload(accept='image/*', multiple=True, description='多選參考圖片')
    picker.observe(_ref_update, names='value')
    return picker

_REF_PANEL = widgets.VBox()
_REF_PANEL.children = (_ref_new(),)
_ref_sync(_REF_FILES)
display(_REF_PANEL)
print('目前圖片：', ', '.join(p.name for p in _REF_FILES) or '尚未上傳')
print('✓ 第 10 節完成：R2V 設定與圖片上傳介面已就緒（尚未生成）。')


11. 生成影片（沿用最後執行的模式 Cell，顯示即時步數進度）


In [ ]:
'''
Serialize notebook generation with optional remote FIFO GPU jobs.
讓 Notebook 手動生成與選用的遠端 FIFO 共用 GPU 鎖，避免同時推論。
不必先啟動 ngrok 或任何 HTTP API。
'''
import threading
GENERATION_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())

'''
Release video workflow caches after the saved output, preserving GPU models only when RAM is safe.
影片完成並儲存後清理工作流程快取；RAM 安全時才保留 GPU 模型。
'''
from contextlib import contextmanager


'''
Use the same cleanup for notebook and FIFO API generations, including failures.
Notebook 與 FIFO API 共用清理機制，發生錯誤時也會執行。
'''
@contextmanager
def _video_memory_cleanup():
    '''
    Start the managed ComfyUI process before submitting video work.
    提交影片工作前，確認 ComfyUI 已經啟動。
    '''
    h3_ensure_service()
    try:
        yield
    finally:
        '''
        Stop ComfyUI after the video is saved, or safely clean a persistent service.
        影片輸出後停止 ComfyUI；若保留服務則使用既有清理機制。
        '''
        if VIDEO_RESTART_AFTER_GENERATION:
            h3_stop_service()
        else:
            import psutil
            import requests
            from pathlib import Path

            '''
            Read available host RAM and the tighter container limit.
            以主機與容器兩者較嚴格的限制估算可用 RAM。
            '''
            available_gib = psutil.virtual_memory().available / 1024**3
            group = Path('/sys/fs/cgroup')
            limit_file = group / 'memory.max'
            current_file = group / 'memory.current'

            '''
            Include reclaimable inactive file pages when the container has a numeric limit.
            容器有明確記憶體上限時，將可回收的非活躍檔案快取納入估算。
            '''
            if limit_file.is_file() and current_file.is_file():
                limit = limit_file.read_text().strip()
                if limit.isdecimal():
                    current = int(current_file.read_text().strip())
                    inactive = 0
                    stat_file = group / 'memory.stat'
                    if stat_file.is_file():
                        for line in stat_file.read_text().splitlines():
                            if line.startswith('inactive_file '):
                                inactive = int(line.split()[1])
                                break
                    available_gib = min(available_gib, max(0, int(limit) - current + inactive) / 1024**3)

            '''
            Discard idle model weights when RAM is tight; otherwise clear only node caches.
            RAM 吃緊時卸載閒置模型；否則只清理節點快取並儘量保留 GPU 權重。
            '''
            unload = (VIDEO_RELEASE_MODE == 'full' or
                      (VIDEO_RELEASE_MODE == 'auto' and available_gib < VIDEO_KEEP_GPU_MIN_RAM_GIB))
            try:
                response = requests.post(BASE + '/free', json={
                    'unload_models': unload, 'free_memory': True,
                }, timeout=15)
                response.raise_for_status()
                print(f'影片快取清理已請求：RAM {available_gib:.1f} GiB，'
                      f'{"模型卸載" if unload else "保留可用 GPU 權重"}。')
            except requests.RequestException as error:
                print(f'警告：ComfyUI 清理請求失敗：{error}')

with GENERATION_LOCK, _video_memory_cleanup():
    import json
    import shutil
    import uuid
    import re
    import math
    import websocket
    from tqdm.auto import tqdm
    TASK = globals().get('TASK', 't2v')
    assert TASK in globals().get('TASK_CONFIGS', {}), f'請先執行 {TASK.upper()} 的參數設定 Cell。'
    config = TASK_CONFIGS[TASK]

    '''
    Keep one submission flow shared across all four tasks.
    四種生成模式共用同一份工作流程提交邏輯。
    '''
    MODE = TASK
    PROMPT = config['prompt']
    WIDTH, HEIGHT = config['width'], config['height']
    SECONDS, STEPS = config['seconds'], config['steps']
    SEED, OUTPUT_NAME = config['seed'], config['output_name']

    '''
    Resolve task-specific reference images without mixing modes.
    依據所選任務使用對應參考圖片，避免不同模式混用。
    '''
    FIRST_IMAGE, LAST_IMAGE = config['first_image'], config['last_image']
    '''
    Keep frame calculation and video encoding synchronized at 24 FPS.
    讓影格數計算與影片編碼使用相同的 24 FPS。
    '''
    FPS = int(config.get('fps', 24))
    assert FPS == 24, 'MiniMax H3／HyperFlow H3 僅支援 24 FPS。'
    
    AUDIO_SHIFT = config['audio_shift']
    QUANT = config['quant']
    assert QUANT in QUANT_SIZES_GIB, f'不支援的 QUANT：{QUANT}'
    REF_IMAGES = config.get('ref_images', [])

    '''
    Select optional reference videos and standalone audio.
    取得選用的參考影片與獨立音訊。
    '''
    REF_VIDEOS = list(config.get('ref_videos', []))
    REF_AUDIOS = list(config.get('ref_audios', []))
    VIDEO_AUDIO = bool(config.get('video_audio', False))
    REF_IMAGE_SIZE = config.get('ref_image_size', 'match')
    ACTIVE_TEXT_MODE = config.get('text_mode', TEXT_MODE)
    assert 0.01 <= AUDIO_SHIFT <= 100.0, 'AUDIO_SHIFT 必須介於 0.01 與 100.0 之間。'

    '''
    Ensure edited GPU 0 memory budgets have been applied to the running service.
    確認調整過的 GPU 0 記憶體預算已套用到目前服務。
    '''
    assert globals().get('H3_ACTIVE_CONFIG') == H3_DESIRED_CONFIG, 'GPU 0 預算已變更，請重新執行第 4 節啟動 ComfyUI。'
    EXPECTED_VARIANT = 'ref2va_pruned' if MODE == 'r2v' else 'fl2va_pruned'
    assert globals().get('H3_MODEL_CHOICE') == (EXPECTED_VARIANT, QUANT) and globals().get('H3_ACTIVE_MODEL') == (EXPECTED_VARIANT, QUANT), (
        f'目前選擇 {QUANT}，但尚未完成下載或服務預檢。請先執行第 2 節與第 4 節。'
    )
    assert DIT_NAME == f'minimax_h3_{EXPECTED_VARIANT}-{QUANT}.gguf', '模型檔名與任務種類／QUANT 不一致，請重跑第 2 節。'
    LAST_OUTPUT = None

    '''
    Confirm the existing local service is still running.
    確認先前啟動的本機推論服務仍正常運作。
    '''
    assert ready(), 'ComfyUI 已中斷，請先重新執行第 4 節。'
    assert MODE in ('t2v', 'i2v', 'l2v', 'flf2v', 'r2v') and PROMPT.strip() and STEPS > 0 and math.isfinite(SECONDS) and SECONDS > 0, (
        '請確認生成模式、Prompt、STEPS 與 SECONDS 設定正確；SECONDS 必須為正數。'
    )
    assert WIDTH > 0 and HEIGHT > 0 and WIDTH % 32 == 0 and HEIGHT % 32 == 0, (
        'WIDTH 與 HEIGHT 必須為正數且是 32 的倍數。'
    )
    FRAMES = max(5, 17 * math.ceil((SECONDS * FPS - 5) / 17) + 5)

    '''
    Respect the underlying ComfyUI node's frame limit, not an arbitrary notebook duration.
    遵守 ComfyUI 節點真正的影格數上限，不另外設定 Notebook 的秒數上限。
    '''
    assert FRAMES <= 3600, f'要求 {FRAMES} frames，超出 MiniMax H3 節點上限 3600；請縮短 SECONDS。'
    import secrets
    RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)
    assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'
    print(f'本次生成 seed：{RUN_SEED}')

    '''
    Build a unique ComfyUI output prefix from the user filename.
    根據指定檔名建立不重複的 ComfyUI 輸出前綴。
    '''
    output_stem = Path(OUTPUT_NAME).stem
    run_name = f'{output_stem}_{EXPECTED_VARIANT}_{QUANT}_{uuid.uuid4().hex[:8]}'
    graph = {
        '1': {'class_type': 'ClipProjLoader', 'inputs': {

            'clip_name': 'qwen3vl_4b_fp8_scaled.safetensors', 'type': 'auto',
            'projection': 'mmh3-4b-ClipProj-v3.1-mlp.safetensors',
            'device': TEXT_GPU, 'mode': ACTIVE_TEXT_MODE}},
        '2': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': DIT_NAME}},
        '3': {'class_type': 'MiniMaxH3SigmaShift', 'inputs': {
            'model': ['2', 0], 'shift_video': 12.0, 'shift_audio': AUDIO_SHIFT}},
        '4': {'class_type': 'VAELoaderMultiGPU', 'inputs': {'vae_name': 'minimax_h3_video_vae_fp16.safetensors', 'device': f'cuda:{VAE_GPU_ID}'}},
        '5': {'class_type': 'VAELoaderMultiGPU', 'inputs': {'vae_name': 'minimax_h3_audio_vae_fp32.safetensors', 'device': f'cuda:{VAE_GPU_ID}'}},
        '6': {'class_type': 'MiniMaxH3ImageToVideo', 'inputs': {
            'clip': ['1', 0], 'vae': ['4', 0], 'prompt': PROMPT,
            'width': WIDTH, 'height': HEIGHT, 'length': FRAMES}},
        '7': {'class_type': 'BasicGuider', 'inputs': {
            'model': ['3', 0], 'conditioning': ['6', 0]}},
        '8': {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'res_multistep'}},
        '9': {'class_type': 'BasicScheduler', 'inputs': {
            'model': ['3', 0], 'scheduler': 'simple', 'steps': STEPS, 'denoise': 1.0}},
        '10': {'class_type': 'RandomNoise', 'inputs': {'noise_seed': RUN_SEED}},
        '11': {'class_type': 'SamplerCustomAdvanced', 'inputs': {
            'noise': ['10', 0], 'guider': ['7', 0],
            'sampler': ['8', 0], 'sigmas': ['9', 0], 'latent_image': ['6', 1]}},
        '12': {'class_type': 'VAEDecode', 'inputs': {'samples': ['11', 0], 'vae': ['4', 0]}},
        '13': {'class_type': 'VAEDecodeAudio', 'inputs': {'samples': ['11', 0], 'vae': ['5', 0]}},
        '14': {'class_type': 'CreateVideo', 'inputs': {
            'images': ['12', 0], 'audio': ['13', 0], 'fps': float(FPS)}},
        '15': {'class_type': 'SaveVideo', 'inputs': {
            'video': ['14', 0], 'filename_prefix': run_name,
            'format': 'auto'}},
    }
    if MODE == 'r2v':
        assert isinstance(REF_IMAGES, (tuple, list)) and 0 <= len(REF_IMAGES) <= 9, 'R2V 圖片最多 9 張，也可以只提供影片或音訊。'
        assert REF_IMAGE_SIZE in ('match', 'max'), 'R2V_REF_IMAGE_SIZE 只能是 match 或 max。'
        picture_numbers = {int(number) for number in re.findall(r'<Picture\s+(\d+)>', PROMPT, flags=re.IGNORECASE)}

        '''
        Refuse to submit a prompt referencing an absent image.
        若 Prompt 指向不存在的圖片，避免送出無效的參考條件。
        '''
        assert all(1 <= number <= len(REF_IMAGES) for number in picture_numbers), (
            f'R2V_PROMPT 引用了不存在的 <Picture N>：{sorted(picture_numbers)}；目前只有 {len(REF_IMAGES)} 張。'
        )
        check_choice('MiniMaxH3ReferenceToVideo', 'ref_image_size', REF_IMAGE_SIZE)
        graph['6'] = {'class_type': 'MiniMaxH3ReferenceToVideo', 'inputs': {
            'clip': ['1', 0], 'vae': ['4', 0], 'audio_vae': ['5', 0],
            'prompt': PROMPT, 'width': WIDTH, 'height': HEIGHT, 'length': FRAMES,
            'ref_image_size': REF_IMAGE_SIZE}}
        for image_index, image_path in enumerate(REF_IMAGES):
            source = Path(image_path)
            assert source.is_file(), f'R2V 找不到第 {image_index + 1} 張參考圖片：{source}'
            assert source.suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp', '.bmp'), f'不支援的參考圖片格式：{source.suffix}'
            node_id = str(16 + image_index)

            '''
            Copy files into ComfyUI's private input directory with unique names.
            將參考圖片複製到 ComfyUI 私有輸入資料夾，並使用不重複檔名。
            '''
            destination = COMFY / 'input' / f'r2v_{image_index + 1}_{uuid.uuid4().hex[:8]}{source.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            graph[node_id] = {'class_type': 'LoadImage', 'inputs': {'image': destination.name}}
            graph['6']['inputs'][f'ref_images.ref_image_{image_index}'] = [node_id, 0]

    '''
    Validate reference media counts and numbered prompt placeholders.
    驗證圖片、影片、音訊張數與 Prompt 的參考編號。
    '''
    if MODE == 'r2v':
        assert len(REF_IMAGES) <= 9 and len(REF_VIDEOS) <= 3 and len(REF_AUDIOS) <= 3, 'R2V 最多 9 張圖片、3 段影片、3 段獨立音訊。'
        assert REF_IMAGES or REF_VIDEOS, 'R2V 至少需要一張參考圖片或一段參考影片；音訊可作為附加條件。'
        assert len(REF_IMAGES) + len(REF_VIDEOS) + len(REF_AUDIOS) <= 12, 'R2V 參考素材合計最多 12 個檔案。'
        tags = {'Picture': len(REF_IMAGES), 'Video': len(REF_VIDEOS),
                'Audio': len(REF_AUDIOS) + (len(REF_VIDEOS) if VIDEO_AUDIO else 0)}
        for label, count in tags.items():
            numbers = [int(n) for n in re.findall(r'<%s\s+(\d+)>' % label, PROMPT, re.I)]
            assert all(1 <= n <= count for n in numbers), f'<{label} N> 編號超出實際上傳數量：{numbers}。'

        '''
        Read reference videos using native ComfyUI video nodes, without extra pip packages.
        使用 ComfyUI 原生影片節點載入參考影格與選用的影片音軌。
        '''
        for index, filename in enumerate(REF_VIDEOS):
            original = Path(filename)
            assert original.is_file() and original.suffix.lower() in ('.mp4', '.mov', '.webm'), f'參考影片不存在或格式不正確：{original}'
            destination = COMFY / 'input' / f'h3_ref_video_{uuid.uuid4().hex[:12]}{original.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(original, destination)
            node_id = str(120 + index * 2)
            parts_id = str(121 + index * 2)
            graph[node_id] = {'class_type': 'LoadVideo', 'inputs': {'file': destination.name}}
            graph[parts_id] = {'class_type': 'GetVideoComponents', 'inputs': {'video': [node_id, 0]}}
            graph['6']['inputs'][f'ref_videos.ref_video_{index}'] = [parts_id, 0]
            if VIDEO_AUDIO:
                graph['6']['inputs'][f'ref_video_audios.ref_video_audio_{index}'] = [parts_id, 1]
        for index, filename in enumerate(REF_AUDIOS):
            original = Path(filename)
            assert original.is_file() and original.suffix.lower() in ('.wav', '.mp3', '.flac', '.ogg', '.m4a'), f'參考音訊不存在或格式不正確：{original}'
            destination = COMFY / 'input' / f'h3_ref_audio_{uuid.uuid4().hex[:12]}{original.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(original, destination)
            node_id = str(140 + index)
            graph[node_id] = {'class_type': 'LoadAudio', 'inputs': {'audio': destination.name}}
            graph['6']['inputs'][f'ref_audios.ref_audio_{index}'] = [node_id, 0]
    for mode, image_path, node_id, input_name in [
        ('i2v', FIRST_IMAGE, '16', 'first_frame'),
        ('flf2v', FIRST_IMAGE, '16', 'first_frame'),
        ('flf2v', LAST_IMAGE, '17', 'last_frame'),
        ('l2v', LAST_IMAGE, '18', 'last_frame'),
    ]:
        if MODE == mode:
            '''
            Require an existing image file for the selected mode.
            確認所選模式的圖片檔案確實存在。
            '''
            source = Path(image_path)
            assert source.is_file(), f'找不到參考圖片：{source}'
            destination = COMFY / 'input' / f'{node_id}_{uuid.uuid4().hex[:8]}{source.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)

            '''
            Register the image loader in the API graph.
            將圖片載入節點加入 API Workflow。
            '''
            graph[node_id] = {'class_type': 'LoadImage', 'inputs': {'image': destination.name}}
            graph['6']['inputs'][input_name] = [node_id, 0]
    NODE_INFO = requests.get(f'{BASE}/object_info', timeout=90).json()
    check_choice('ClipProjLoader', 'device', TEXT_GPU)
    check_choice('ClipProjLoader', 'mode', ACTIVE_TEXT_MODE)

    '''
    Verify the H3 sampler against the specialized node or the core KSampler options.
    優先用目前採樣器選單驗證；必要時參照核心 KSampler 選項。
    '''
    check_choice('KSamplerSelect', 'sampler_name', 'res_multistep', ('KSampler', 'sampler_name'))
    check_choice('BasicScheduler', 'scheduler', 'simple', ('KSampler', 'scheduler'))
    if MODE in ('i2v', 'l2v', 'flf2v'):
        frame_inputs = NODE_INFO['MiniMaxH3ImageToVideo'].get('input', {}).get('optional', {})
        expected = ('first_frame', 'last_frame') if MODE == 'flf2v' else (('last_frame',) if MODE == 'l2v' else ('first_frame',))

        '''
        Reject an incompatible core node implementation before generation.
        如果核心節點不支援這些欄位，在生成之前先停止。
        '''
        assert all(field in frame_inputs for field in expected), f'MiniMaxH3ImageToVideo 缺少 {expected} 輸入。'
    if MODE == 'r2v':
        ref_inputs = NODE_INFO['MiniMaxH3ReferenceToVideo'].get('input', {})
        ref_fields = set(ref_inputs.get('required', {})) | set(ref_inputs.get('optional', {}))
        assert 'ref_images' in ref_fields or any(name.startswith('ref_images.ref_image_') for name in ref_fields), (
            'MiniMaxH3ReferenceToVideo 缺少 ref_images 輸入，請更新 ComfyUI。'
        )

    '''
    Confirm the installed ComfyUI version supports all reference-media loader nodes.
    確認目前 ComfyUI 具備參考影片與音訊所需的原生節點。
    '''
    if MODE == 'r2v':
        required_nodes = {'LoadVideo', 'GetVideoComponents'} if REF_VIDEOS else set()
        if REF_AUDIOS:
            required_nodes.add('LoadAudio')
        missing_nodes = required_nodes - set(NODE_INFO)
        assert not missing_nodes, f'ComfyUI 缺少參考媒體節點：{sorted(missing_nodes)}，請更新 ComfyUI。'
    for node_id, node in graph.items():
        '''
        Examine each declared input for possible node-reference links.
        檢查節點輸入中是否包含連到其它節點的參照。
        '''
        for value in node['inputs'].values():
            if isinstance(value, list) and len(value) == 2:
                assert str(value[0]) in graph, f'節點 {node_id} 連結到不存在的節點 {value[0]}。'
    for node_id, node in graph.items():
        schema = NODE_INFO[node['class_type']].get('input', {})
        accepted = set().union(*(set(schema.get(group, {})) for group in ('required', 'optional', 'hidden')))
        unknown = set(node['inputs']) - accepted

        '''
        Allow dynamic Ref2VA image slots declared by the live autogrow parent.
        當節點宣告 ref_images 動態欄位時，允許自動展開的圖片插槽。
        '''
        if node['class_type'] == 'MiniMaxH3ReferenceToVideo' and 'ref_images' in accepted:
            unknown -= {f'ref_images.ref_image_{i}' for i in range(9)}
            if 'ref_videos' in accepted:
                unknown -= {f'ref_videos.ref_video_{i}' for i in range(3)}
            if 'ref_video_audios' in accepted:
                unknown -= {f'ref_video_audios.ref_video_audio_{i}' for i in range(3)}
            if 'ref_audios' in accepted:
                unknown -= {f'ref_audios.ref_audio_{i}' for i in range(3)}
        assert not unknown, f"節點 {node_id} ({node['class_type']}) 不支援的輸入：{sorted(unknown)}"
    print(f'{MODE.upper()} | {WIDTH}×{HEIGHT} | {FRAMES} frames | {STEPS} steps | {EXPECTED_VARIANT}/{QUANT} | Audio Shift {AUDIO_SHIFT}', flush=True)
    client_id = uuid.uuid4().hex

    '''
    Track WebSocket availability separately from workflow success.
    個別追蹤 WebSocket 可用性，不影響實際生成流程。
    '''
    ws = None
    try:
        ws = websocket.create_connection(f'ws://127.0.0.1:{PORT}/ws?clientId={client_id}', timeout=5)
        ws.settimeout(2)

    except (OSError, websocket.WebSocketException):
        '''
        Mark event updates as unavailable and retain an elapsed-time display.
        停用事件更新，改成單一進度列顯示經過時間。
        '''
        ws = None
        print('即時步數事件不可用；將顯示階段與經過時間。', flush=True)
    response = requests.post(f'{BASE}/prompt', json={'prompt': graph, 'client_id': client_id}, timeout=120)
    if not response.ok:
        if ws is not None:
            ws.close()
        raise RuntimeError(response.text[:5000])
    prompt_id = response.json()['prompt_id']

    '''
    Start timing the actual queued job.
    開始計算已提交生成工作的耗時。
    '''
    start = time.monotonic()
    pbar = tqdm(total=STEPS, desc='排隊／載入模型', unit='step', leave=True)
    item = None
    last_history_check = 0.0
    phases = {'1': '文字編碼器', '2': '載入 GGUF', '6': '建立影片條件',
              '11': '採樣中', '12': '影片解碼', '13': '音訊解碼',
              '14': '合成影片', '15': '儲存影片'}

    '''
    Wait until ComfyUI reports a final history record for this task.
    持續等待 ComfyUI 回傳此任務的最終執行紀錄。
    '''
    try:
        '''
        Keep processing until success or an explicit ComfyUI error.
        持續處理直到成功或 ComfyUI 明確回報錯誤。
        '''
        while True:
            '''
            Measure real wall-clock time instead of estimating model progress.
            計算真實經過時間，不將時間誤當模型百分比。
            '''
            elapsed = time.monotonic() - start
            message = None
            if ws is not None:
                try:
                    message = ws.recv()

                except websocket.WebSocketTimeoutException:
                    pass

                except (OSError, websocket.WebSocketException):
                    '''
                    Release the unusable client connection.
                    關閉失效的 WebSocket 連線。
                    '''
                    ws.close()
                    ws = None

            '''
            Only JSON text frames contain progress data; preview images are binary.
            只有 JSON 文字事件含進度資料；預覽圖片為二進位資料。
            '''
            if isinstance(message, str):
                event = json.loads(message)

                '''
                Read the event payload without assuming optional keys exist.
                讀取事件資料，並避免假設選填欄位一定存在。
                '''
                data = event.get('data') or {}
                if data.get('prompt_id') == prompt_id:
                    if event.get('type') == 'executing' and data.get('node'):
                        pbar.set_description(phases.get(str(data['node']), '執行中'), refresh=False)
                    if event.get('type') in ('progress', 'progress_state'):
                        progress = data
                        if event.get('type') == 'progress_state':
                            progress = data.get('nodes', {}).get('11', {})

                        '''
                        Ignore progress emitted by loading or decoding nodes.
                        忽略非採樣節點的進度，以免誤顯示百分比。
                        '''
                        if str(progress.get('node', progress.get('node_id'))) == '11':
                            value, maximum = progress.get('value'), progress.get('max')
                            if isinstance(value, (int, float)) and isinstance(maximum, (int, float)) and maximum > 0:
                                pbar.total = int(maximum)
                                pbar.n = max(0, min(int(value), pbar.total))
                                pbar.set_description('採樣中', refresh=False)
            if elapsed - last_history_check >= 4:
                last_history_check = elapsed
                if not ready():
                    '''
                    Preserve recent server logs for actionable diagnostics.
                    保留最近服務紀錄供後續除錯。
                    '''
                    raise RuntimeError('ComfyUI 服務中斷：\n' + LOG.read_text(errors='replace')[-4500:])
                item = requests.get(f'{BASE}/history/{prompt_id}', timeout=60).json().get(prompt_id)

                '''
                Stop polling once the job has a final record.
                任務產生最終紀錄後停止輪詢。
                '''
                if item:
                    errors = [m for m in item.get('status', {}).get('messages', []) if m[0] == 'execution_error']
                    if errors:
                        raise RuntimeError(str(errors[-1][1])[:5000])
                    assert item.get('status', {}).get('status_str') != 'error', (
                        f'ComfyUI 工作失敗：{item.get("status", {})}'
                    )
                    break
            pbar.set_postfix_str(f'{elapsed / 60:.1f} 分鐘', refresh=False)
            pbar.refresh()

            '''
            Avoid tight polling if the WebSocket connection is unavailable.
            WebSocket 不可用時稍作等待，避免過密輪詢。
            '''
            if ws is None:
                time.sleep(1)
        pbar.n = pbar.total
        pbar.set_description('完成', refresh=False)
        pbar.set_postfix_str(f'{(time.monotonic() - start) / 60:.1f} 分鐘', refresh=False)
        pbar.refresh()

    finally:
        '''
        Always free the live progress widgets and connection after this job.
        無論任務成功或失敗，都釋放進度列與 WebSocket 連線。
        '''
        pbar.close()
        if ws is not None:
            ws.close()
    files = [p for p in (COMFY / 'output').rglob('*') if p.is_file() and p.name.startswith(run_name)]
    files = [p for p in files if p.suffix.lower() in ('.mp4', '.webm', '.mov')]
    assert files, f'工作流程完成但未找到影片：{item.get("outputs", {})}'
    source_video = max(files, key=lambda path: path.stat().st_mtime)

    '''
    Save the user-visible output in Kaggle working storage.
    將成果儲存至 Kaggle working 目錄。
    '''
    OUTPUT_DIR = ROOT / 'minimax_h3_outputs'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_base = OUTPUT_DIR / f'{Path(OUTPUT_NAME).stem}_{EXPECTED_VARIANT}_{QUANT}{source_video.suffix}'
    output = output_base

    '''
    Number duplicate filenames deterministically.
    依序為重複檔名編號。
    '''
    for index in range(2, 1000):
        if not output.exists():
            break
        output = output_base.with_name(f'{output_base.stem}_{index:03d}{output_base.suffix}')
    shutil.copy2(source_video, output)
    print(f'完成：{output}｜耗時 {(time.monotonic() - start) / 60:.1f} 分鐘')
    LAST_OUTPUT = output
    print('✓ 第 11 節完成：影片已生成；請執行第 12 節預覽與下載。')


12. 影片預覽、下載與 GPU 使用量


In [ ]:
'''
Display the most recently generated video and its download link.
顯示最近一次成功生成影片的預覽與下載連結。
'''
from IPython.display import Video, FileLink, display
assert globals().get('LAST_OUTPUT') is not None and LAST_OUTPUT.is_file(), '請先執行第 9 節生成影片。'
display(Video(str(LAST_OUTPUT), embed=True, width=480))

'''
Provide a clickable Kaggle file download link.
顯示 Kaggle 影片檔案下載連結。
'''
display(FileLink(str(LAST_OUTPUT)))
# 顯示指定兩種模型元件的 GPU 記憶體用量，不依賴 nvidia-smi。
import torch
for gpu_id in sorted({MODEL_GPU_ID, TEXT_GPU_ID}):
    free_bytes, total_bytes = torch.cuda.mem_get_info(gpu_id)
    used_gib = (total_bytes - free_bytes) / 1024**3
    budget = GPU0_MODEL_GIB if gpu_id == MODEL_GPU_ID else GPU1_MODEL_GIB
    print(f'GPU {gpu_id}：使用 {used_gib:.2f} / {total_bytes / 1024**3:.2f} GiB（參考預算 {budget:.2f} GiB）')

print(f'✓ 第 12 節完成：影片預覽與下載連結已顯示；{LAST_OUTPUT.name}')


13. 擷取最後影格（選用）


In [ ]:
'''
Count decoded video frames to identify the final frame.
計算影片影格數，找出最後一個影格。
'''
import json
from IPython.display import FileLink, display
from PIL import Image
assert globals().get('LAST_OUTPUT') is not None and LAST_OUTPUT.is_file(), '請先執行第 9 節生成影片，並確認第 10 節可正常預覽。'
assert shutil.which('ffprobe') and shutil.which('ffmpeg'), '找不到 ffprobe 或 ffmpeg；請先安裝 FFmpeg。'

'''
Build a PNG path next to the generated video.
在生成影片旁建立最後影格 PNG 檔案路徑。
'''
LAST_FRAME_PATH = LAST_OUTPUT.with_name(LAST_OUTPUT.stem + '_last_frame.png')
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-count_frames', '-select_streams', 'v:0',
     '-show_entries', 'stream=nb_read_frames', '-of', 'json', str(LAST_OUTPUT)],
    capture_output=True, text=True, check=True,
)
last_index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1

'''
Save the exact last frame as a PNG for the next I2V clip.
將最後一個影格儲存成 PNG，供下一段 I2V 使用。
'''
subprocess.run(
    ['ffmpeg', '-y', '-loglevel', 'error', '-i', str(LAST_OUTPUT),
     '-vf', rf'select=eq(n\,{last_index})', '-frames:v', '1', str(LAST_FRAME_PATH)],
    check=True,
)
display(Image.open(LAST_FRAME_PATH))
display(FileLink(str(LAST_FRAME_PATH)))
print(f"下一段：TASK = 'i2v'，FIRST_IMAGE = '{LAST_FRAME_PATH}'")
print(f'✓ 第 13 節完成：已擷取最後影格；{LAST_FRAME_PATH.name}')


14. （選用）啟動 ngrok 與 FIFO 遠端 API


In [ ]:
'''
Require the first configuration cell before enabling the optional remote API.
選用遠端 API 啟動前，必須先執行第一格共用參數設定。
'''
assert all(name in globals() for name in ('REMOTE_PORT', 'REMOTE_SERVICE_ID', 'REMOTE_ROOT')), '請先執行第 1 節共用參數設定。'
import zlib
import math
import re

import queue
import copy
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

REMOTE_KIND = 'video'
REMOTE_FIELDS = ['task', 'prompt', 'seed', 'width', 'height', 'seconds', 'fps', 'steps', 'ref_image_size', 'audio_shift', 'quant', 'video_audio']
REMOTE_IS_LTX = False
REMOTE_SCRIPT = zlib.decompress(b'x\xda\xd5}ks\x14G\xb6\xe0w~E\xedDLt\xf7XjK\x02{\xc6\xech6\x18\x10X;6p\x11x\xee]\x86\xe8(uWKe\xfa5]\xdd\x12\x1a]EH\x18!\x81\x10\x92\r\xc8\x16\x88\xa7\x01k\xc0H`^B\x08\xf1\xe1\xfe\x93\xbd]\xfd\xf8\xe4\xbf\xb0\xe7\x91\x99\x95U]-\x84g\xf6n\xecL\x04VWWg\x9e<y\xce\xc9\xf3\xceH$\xb2\xa37[\xc8\x17KFi\xd022v\x7f\xd1,\xda\x96c\xe4,+e\xa5\x8ct\xbe\x08_\xd8\x8e\xe1X\xc9\x92\x9d\xcf\xc5w\xb8\x17V\xdd\xc9{\xd5\xa5\x1f\xab+\xcfk\xcb3\xee\xc6\\\xf5\xdcxci\xbcv\xf5\x8c;\xf5\x06>\xba\xeb\x0f\xffs\xfc\xf4\x8e\x08\x8cl\xf3\xc8_:\xf9\x9c\xfc\xdb\x19,\x97\xec\x8c\xfcT.\xdb)\xf9w\xd1\x92\x7fe\xcd\xd2\xa0\xfc{\xd8\xeaw\xf2\xc9\x93ViG\xba\x98\xcf\x1a\xa5\xbf\xa6\xb2q\xb3\\\xca\x1b\xe2{|\xb0\x83&\xeb\xb32\x00#-#e9v\x11\xa0/\x99\xceIc\xd8.\r\xe6\xcb%#?d\x15\x87\x8bv\xc9\xce\r\x18v.e\x15,\xf8\'W2\n0n\xa1\xe4\xc4\x8d\xbd\x83\xf9\xbcc\xd1\x00\x03V\xce*\x9a\xb8`\x1e\xc3\xce\x19}\x8c\x00\xa3+\xbe\xa31\xb1V\xbd4\x85(\xf8\xf1v\xed\xf2\xcd\xea\xf4<,\xbe\xf2\xfa\xb5;\xb3\xf8\xf3\xc6\x85\xca\xdal\xfd\xfeYw\xf5\xa1;\xf9\xc2]\xf9\x8a\x9f\xc3\x0b\xc6a\x9a\x08pc\xb8K\xcb\xb5\x1f\x7f4\xba\x8c\xda\xea8\x0f\xf6\xf3\xc6\xd5H\xa9k(\xd2fDl\xfeO:\x93\xe6?\x8a\xf0\x1fZ\xe0\xd1=}\x7f2\xba\x8d\x81L\xbe\xdf\xcc8\xd1X|\xc0*E#\xf8\x14_\xc3_\xc7\x18\x13G,3E\xcbp\x08%\x02\x11\x11\x07\x96j\r\xd9\xf9\xb2\x93\x191\xacSV\xb2\x8c_%\xf3\xb9\xb4=P\x16\xabMZ\x99L|G}e\xdc\x9d[\xc0}\x9dXc\xf8\xdd\x97?\xb97_\xd6o_hL\\\x84\xb5\xd4\x97\x1f\xb9+W\x8d\xbd\xf0\xb6\xdck\xd3q,\xd8\x0f\x02\x12\xd0\x15\x02eb\xef\xa1\x83\xfb{\x0f\xf4\x01\xb4\xa3c\xb16#\x1d\xa9?\x9cq\'\xa7y`c\x14\xdf\x89\x97\x0b\x05\xab\x18\x8d\x8d\x19HOs_U\xaf\xac\xf9\xe7\x8a\xc8%:\xf9\xcc\x90\x15\xbaJ}s\x1d\xab\x84\x1b\xee\x00\xdd\xce-\xb8\x9b\xdf\xeak\x82\x19j\xb3\xcb5\x80\x81\xe6\x91\x0ba\x84\x00\xa2u\x98\x8f\xe3\x87\x13<\xf7\x9f,\xab`\xe4s0q\xb9?k;\x0e\xe2-\x9d\xc9\x0f\x03m\x9bHuf\xb2\x98w\x1c\xc3\xccd\x80}\xcaE\x02\x0b\x01\xb8v\xad\xb6\xbc\xc2\xf4R]\xbe\x8d\x9c2\xf9\xa4vy\xd9\x9d\x07\x92\x19\xaf\xbc~\xe3\xbe\xbcWy\xb3T}>\x01lU\x9d\x9b\xaf\xac\xdfm\x9c\x9e\xabo\xacJ\xc0>?\xb4\xafG\x80\xc5\x80\x1c\x13\xd4\xca\x14l8\xa5<N\x0f\xc8o\xc2\x8a\xd8\xd7\xca\x9b\xb70a\xed\xda\x8a{n\x96q\xa0!\xd7\xa8\x7f\x7f\xdbG\xa64\xc7\xe1#\x87>?|\x14\xa6e\xb4\x1c\x8f\xf0d\x91\x13~\x10\xd4lI37d::\xde\xef}\x0bs\x06\xf0^\xdfXs\xa7\xd6\xdd\xc7\xeb\xee\xaa\xc2\xfb\x9f{\xf7\x1d\xfd\xb4\xcd\xf8\xb4\xa7\xf7\xc0\xa7\xfa\x84\xc3v\xaa4\x189\xd1\xa6\x1e\x0cZ\xf6\xc0`\x13\x04\xb8\xccv\xa7`%\xed\xb4\x9d4R\x92\xa0\xcd\\\xcap\xccl!\x83l\x9f\xcc\x97s\xa5\x16 \xd5~\xf8\x06\x89mz\xaaz\xf1vu\xf9\xfb\xea\xa3{\x1aM\xf4\xf5\x00\x1d\xec\xebk3\xfa\x8e\xf6\x1c\xee\xd3\x80\x03\xb1\x98\xcf\xa5\x1c\x1d<\xa7d\x15\x1c\t\xdd\x9eB\x01\x98\xad\x19>\xc7BB\x01\xd8@2\x15@8\xa5\xed\x8c\x953\xb3V+\xe0\x96W\xdcG\xf3\x00\x1c#\xae\xfa\xe0\xb2;?\xeb\x01\xd7\xb3\xaf\xcd8t\xec\xe8\xe1cG\x13\x07\xf7|\xde\xe3\x83\xcfJ\xe9\xc0\xf1t\t\x9cJ\x82\xa8X\xc9\x07`\xd1J[E+\x97\xb4@\xd0\x9a\x03p H1\x9a\xb5O!*\xb3y\x10\xb2@Q\x9b\xd7\xab\x97\xae\xea\xd02\x8d\xb9\x8fg\xabS\xe7\x80\xb1\xea\xe3_\xb9K\x0b\xb5s \xe1@v\xbcu\'gA>\x02\xc93\x07T_\xbe\x84\x97\xe5B\xf6\xf7\x1e\xe9;\x9a\xe8\xfd|\xcf\x81\x9e6\xe3\xb3=\xf2om9i\xbb\xe8\x94\x12\x04\x91\xbe\xaa\x8c\xe9=\r\xe2=(\x1d\xccr\xca\xce\x1b\x8e=\x905\x81a\xedtI\xad\xccJ\xf1\xe1\x108\x00\x06\x8afaP\xf2N\xf5\xd1]\xb5)\x8d\x9bO\xeb\xcb\xe7\x8d>\x1a\xaa\x0f\x87b\xe9\xef\xbe\x9d\xac\xbc]\xa9^~\xc5\xbcn\xfc9_<\x89\xf2A.S\xc9\x90t\x11\xb6\x018&\x93,g<j\x1d\xb2SV\xde\x00\xd4\xe7S\x08\x8d3\x92K\x0e\x16\xf39\xfboH1%\xa3k\x97\xb1\xffp\x1f\xca\xe8K\xee\x9b\'\xd5[\x1b$"\xa7k+\xdf\x02y\xc0\x13@u\xed\xe5r\xed\xce\x86d\xf65\xc06\xf25\xffP\xe1\x9a\xc8\xd8\xce\x95\xa2\x8cD\x16\xd3i\xa0\xdc6x3\x16\x93\xf2\x9c\xde\xeb\x86Gp\xc4|n\xe7\xec\xcf\xcdS\xc6\xa7;\x7f\xde\x98\xfbt\x04\x04\xf5~\x14{\x9f\xee4\xdc\xaf&\xab\x97W\xabs\xcf\xbcY@P\xef9\xb6\xaf\xf7P\xa2\xef\xd3\xde\xfd:?\xd3\x06$\x08\xf5\xef\xde\xad\x03\x07\x8e\xed7\xfeZ6s%\xfbo\x8c"\xd4H\x90\xf828\xa0c\x03\xb7\xe5\x92#Fr\xd0J\xa2\x90\r\xecR\xf5\xc2\x14\x887\\=\r\xd4\x98\x9as/,\x00\xce\x80\xf4p\xaf^\xdf\xab\xddY\xaf\xbc\xbd\x8e\xc4xc\xa6\xfa\xe0N\xf5\xe6=\x90\xc5\xf5\xa9g\x12M\xffrl\xcfA\x1dx\x02\xc5\xe3\x9c/Q\xe3(\xe7\x1c8\xb5@\x19\x01\xb0}\xa0"\x979F\xbf\x05 [\xb0u%\x0b\x04\'n\xa9G\\\xf1\x1dL# \x8f\xe1 \xac^\xbe\\\x9d~\x00\x14\xc4\xb8D\x12\xd3\x00\x06\x8e\xaf-?\t\x1c\xb7\x0c\x1f\x88|\xfa#\xd1\xd7\xfb\xbfz\xfa\x12\x07z\xff\x88\xa7\xab>\x10\x7f\x0fZ\xc6(\xfd1&\x8e\xd1\xcf\xf2\xa0)(Vo\xcf\xe7`\x1b\n&\x92e\xc9*z<o\xa6\xd3\xa8\xfd\x00\xe8D\xb2\xed\xfd\xa6#\xb6\xc8\x91\xca\x82q\xa4\xeb\x0b\xc3}<\x8d\xbcO\xc7\xa9\xe0\x857O\x1a7\xdf\xba\x17o\x82\xb6\x84\xeb\xb9\xbf\xe0>\xde\x14tK\x12@\xae\xe7H\xcf~\xe6vO\xba2I\x02t\xcc\xdaH\x99\xc7O\xc4|\xca^\xbe\x80X43\x9a\xb8"\xf6qX\xee\x97\xe0_3\x83G5\x11\x9dT\x02@V!g\x90\x82\x81\xf2\x89\x98\x06\xb8\x87\xb5\x01fk\x1d\xae/z\xf7\xf5\x1cB\xb82@n\xd1 p<!\x03\x17\xa3\xf7\x89\xee[\xbeO\xa0\xa8\xf7il\xfe\x05\xfc\xa0?\x9f\xcf\xf8~@\x83\xf3O\xe0\x17\xfbA\xa3\xb2bA-\xcf[\xbb\x03B\x02V\r\xb4e\r\x8c\x18\xb4\x9b\xac\xc0[\xa0\xcf\x0f\xd3\x16\x15@\xb7F<<\xa8.\x9d\xa3\x07B^+I\xbd\x06\x92k\xb3\xf6h\xa1v\xe5^\xd3\xde\x10}\xb5\xda\x9f\x04\xce\x8e\x9a(\xa8\xef\xc9\xc1H\xacI+\'\x91f\x15\x01^\x07V\x85\\\xab@\x95P&\xcb\xc5"\xaamHZR\xcffu\x85\xa9\x05)\xb9G\x0cS\xdf8\x07\x86\x88\x1f\xd2={\x8f\xf6~\xd1\x938\xda\xf3\xafG\x13B]\xd2a-Y\xa7J\t\x14\x1e\x00\xa6zG\x00\xfa9B\xcd\xa8*g\xad"\x1e\x81fn\x00m\x89d\x86\xd4\xba~\x96Pf\xb2T\x06\x82\x13\xb2\xf0\xd3\x9d$\xfeI\xfa\x1b9\x189\xbe\xa3\xbaz\xb96\xf9C\xe8\x0b\xa8\xee\xbf\xbe\xe1\xae|\xef~}\x1e\x96\x02\\\xe2\x8eo\xd4V\xcf\xb8K\xb3\x8d\xf1\x9f@\tf\x11\x14\xe0\xf1\x8exG\xa7\xf1\xfbnC\x17\xa7\xf0\xb1\xb3\x03\xbe\x00t\xeb\x8f\xe1\xe8i\xdc\x9a\xae\xbc\x9e\xa9.l\xf0\xef\x80\xac\xf9M\xa3\xf2j\xa6\xb1p\xc9S\xa1{@l\x81\\\xc2S\x0fVw\xe0\xf01\xa3\xc3\xc8Z\xd9|q\xc4\xe8/\xa7\x00_\x8e1h\x82^\xd0oYp0\x81\x84\xb6\x91\xe7\xf3Lr\xe5\\\x8e\x0e\'\xab8d\'a\xd1 G\xeb\x0ff\xeb\x0f\xdeV\xaf<c\x13A\x8cX_\xfe\xaez\xf6E\xe3\xe1\xe5\xc6\xady8\xa3\xd0\x8e \x05\xc7\x9d~,6vi\x16\x04u`\xc9\x01\xeb\xe1\xd3\x9d\t\xb1\xb3\xac\x8eGbx&\xc1\xd3}=}\xbdGz\xf6\x89\xc7\x80\x0c\x9eT\xcdU_9_\xbd\xf6\x0cD\x11X\x1a\x8d\xa9\xd9\xea\xc2c66\xd0\xfa\xda\x85\xdb\xe1^\xb9\xe9\xce\\1\xf6\xe6\xb3\xe9\x91c\xbd\x1er\x0e\x83\xa5\x84\x84\xa8\xe9\x01\xa0:\xda\x19\xa6\xd2A\xb0\x13s~Y?\x0cJo*?\x9c\xcb\x80D\x15\xaa]\xd1\x1a\xc0\xe3\th\'n\xf4\x9c* \x1b\x98\xc6\x11+\xdd\xf5\xc5\x1e\xa0\xaa\\\xde\x86\xa52\x93\x0e\x0fZ9?+\xb7\x13K\t\xe3\xd3Q\xc7"p.(\xa8s\xd7|\xe7\xc2\xe6\x05X \xef<ZS+\x17\xe0@\xa9\xac\xcd\x00\x7f\xe0\xe6\x8b\xa5\x19|\xc6\xd5\x97_\xbbg\xcf\x93\rJ"\x00XL\n\xef\xea\xe2\xe9*\x98\x05$\x0e$\x98\xee\xc5\xd7\xee\xe2\x03\xfe\xa9\xdc\xa2\x9e\x7f=\xdc\xb3\xf7( \xfd\x8b=Gz\xf9tD!\xd05d&\n@\x17\xa0o\x1av\xda`\xee\xebf\xf3\xd5\xb0@n\xa1I\xab\xbd$\x85\xd8_\xcb`\xaa\x83\xe0+1\xf3\xf1\xd1\x9e6\xb36\xe0\x05\xb1\x88\x0f}\x98\x06\x12\xcc*^eA\x81\xfa\xe4\xe2i\xa6A\x06\x16\x14\xe6\xc6\xed\x1b\xee\xdc\xf9\xc0\x91\xcf*h\xe3\xc2j\xfd\xc1\xdd\xeaO\xb4\xd4\xfd\x9f\xe1J\xab?\xc2\xab\xb3\xef$C\\\xd6g\x89\xbd\x9f\x1e\xea\xdd\xdb\xc3T\x18\r\xe2\xa3\x8d\xcf\xda\x18A\xdf\x92\x90i\xa0w\x8c\xd0fDw\x18\xf0\xbft\x84Y\x85\xe5\xa1!\x8ep<`\xdf\x9cu\x1f_\xad.=\xd07\xbd:\xbd\xc0<\x05\\\x00\xf2\x04\x96\xa4\x9b\xd9\xca\xef\x80\xe7\x9d\xe4\x02\xa2\xfb\x98O, r\x87\x85\xe2\xaa\xac\x13\xb0i\x87\xcd\x11\x87\xf1\x0f\xaa\x8d.\xb2\xf5=\x92\xf2@\xa9\xbe\x06[,8\xa7\xb6\x92\x80\x82\xe3\xd7\xbc\xc4\x16\xec\xeb\x956M7\xe0!\x0bB5k\x9eJ\x0c\xeeL\x8c\x06\xb16\xd6.\x10\x13\x1f\x18(\xa7\xf1 \x92j\x9d\x98Z(\xeeD\x19\xa0\xc0\xb2\xea\x04:\n\xcf\xab\x84D\xfd\xe5\xd7\nI\x9e@\xd83\x94\xb7A\xa3\x18\xcc\x0f\xa3\xdc3Q\xb7\xc8 \xbf:\xe5\x0cXo\xe8\x9a2s\x86e\x16AJ\x16\r\xa7\x9cLZ\x8e\x93.gPR\x82\xfa\x04R\x00~\x936\xed\x0c\xe0\x16\x0f6\xa4B\xdd\x81\xe4\xde}R\xbd\xf2-\xb3rui\x1c\xfe\x00\xa8\x00-\x95\xd7 T\xef6n\xaf\xd6\xee\xae\xd7\xa7\xcf\xb3\xaa"QD&\x12\xdb}\xc0\x84\x07A\xcfaX\xf7\xe2\x91W\xcc\xf2\xa1{\nd\x10\x82\x9c\xc9\x83\xa1!\x056\xc9\x14\x10g\x19)\xc9\xe5\x8e!\x9d\x80.JR\x11O(\x00\xf1\xef7\xab\x17\x97\xeb\x0f\x05Q\x01@\xd5G\xdf\xbbkk\x8d\x89\x99\xca\x9b\xa5\xc0n\x15A\'\x19\x89\x02\xe1F\xa4\xd8\x01!\\Y{T]x\xc9\xf8\x85\xf1C\xe5\xb0\x87\xe8/\xcc\x8c\x9d\x02\xad h\x86\xa1`h\x13\xae\x8e6\xcf\xa4\'k\xbb\x8dx\xad\x90w\xc0\x80\x83\x13K\x1a\xffp\x16\xd3y\xaa\xfb\\\xfes|B:8&tK\x1fD\x05{\x00\xaa\xdf\xad\xba\xf3\xf7A4V\xafL\x07VG2\rt\xedh\xc0U\x97\t\xf1\xd81\xfb\xb3\xf7$\x0ez\x8e]\x88\xf2#\xf6 \xfc\x01N)\xfc\x84\xae\xce\xb8\xed\xa4\x81\xaeKVT\xb8\x19\xc4\x8b\xfc\x01_\x95\x82\x00\x1df\xbcO\xad\x96\xc4\xa3\xa3\xdc\x97?g\x0f\x0f\xec\x19\xfc\xf0\xe7\x8dk\xf21\x9f\x17\xb5\xd3\xeb\xf0\r;:\x94\x04\x902\xd9\xc3\xa7\x9d\xb5r\xe8\xe6B\xa7\x96=\x90c=`gW\x9bg \x00\x91\x17K\xe88\xb0I\x19w\xf2\x992\xedZ\xc6\xce\xda\xe8qY\xfd\xb1\xf1\xf0;\xf4\x12\xcb\t+k\x97k?\xdew\xe7\xa7a\x1c\xf2\xf5\x8d\xcfzF\xc3\xdc}\xf7\xeeBcq\xde\x9d~Q\xff\xe1\xfb\xea\x8dyw\xfd>\x88\xb5\xc6\x95\x970\x10\xe8W\x81}!\xcf\x91B\xa9p \xc9\x8f\xfc\xe5\xafq\x1a\x10!\xbeW\xd4C\x85_~\x19\xd1\'^\t\xa0\t\xa0\x06\xfa\xf0\x81\xec\xc3\\\xbe\x9c\x93:\xf9_\xcb\x96\x83\x8a\x95\xf2D\x95\x0b\x885\xc0\xd4\xa7;\xdb\x93\xb0]\xf0\xb4\x1fDH\xe7os\x1f|$]\x01\xec\x9f\xaal^G\r\x12\xecj\xd4\xa3\xf8\x85\xfa\xfd9\x97\x9c(\xee\xe3)a\xd5\x12\xb9\xba\xf3_W\xd6\xce\xa3\xb7\xf6\xca3\x00\x13\x10Z}zE9\x06\x94\xb5\x7f\x04D(Z$ <\xa3\x1f\xb5\xc1\x98\xc6o\x98\xf6\x92\x96\x9d\x89J\xc2\x83\x87h\xef\xb7\x1b\x1f\xc5\x8c\x0f\xe1\xa5\x98\xf1\x01\xfc\xa9\xdcD\x05\xa9\xc9\xc3*\xadbf\x04\xf9O\xf29\xea\xbe`\xaf\xf3:h\xd7\xdb\xe0\x99\xa0\x8c~\x1b\x14}\xd0+\xe1\x81\x05&\xceI\x8dC\x1b\x13\xcf\xdd\x95i5\x0ck\xc8\xb5\xa5\x9bH\xb0\x80b\xb9\x10X#\xd0\x83N\x1e\xc2syP\x0e\xa9\\x\xfcj\x80D\x04\x02@m\xde\xf9qG\x07\xf9\x9e\xefOT\x9f\x9c6F\xf9\x9b1\x86\xdcA)\xf5b\xd2\x9dZ7<g\x87\x00\x8a\xc7\xa5\xdf\x03\x1f!\'\xae\xad\xd4n>\x92\x8c\xe6I\xb0\x03,\xb4,\x94\xf8\xc0\n\x83@.\xed\x80\x01\xf6\xf9y\xfa^{\'JaE(\xd2(\x13\xeb\x82oY!s_\xaf\x83Q\n\x12\x13iA\x8c\xd4\xb8\xba\x0cb\x99\x9d\x82\x81X\x8bc%\x8b\xa0\xba\xfb]{\xe4\xac\xc6\xd9\x0bh~\xc1\x89\x14\x16\x13!\xd3GbS\xf8o\xd1\x9a\x99S\x87T\xf5\xc9:\x80\x06\x07\x93;>\xc3\x93\xb3\x0e\xa2"\x1e\xbe\xdd\x00\xa3\xdf\xfda\x86\x97\xa3,\xc9c\x07\x13\xe8\xae\x04J\x14\x80\xc6\xc1\xcaJ\xf5[\xa0#D\xbb~\xf3\x9b\x9d]1T\x1e\xf9\x95nD\x02i\x8e\xe8\xa6\xc2g\x01\x01\x05\xa7\xb0r\xbc\xd0\xf2\x84\xab\xa5\xbflgRrIvNZ\xc7R\x9d\x01L/-\x0b\xb4\xd2\xc9\xa6;\xde\xe1\xf4C&\x9b\x9c\x16G\x07-\x93M\xb4\xa0Y\x86\xd4\xa4\x16\xf4{\x83\xc0\x07\xe9O\x1f\xfd\x86\x98Q\x9fzj\xec\xea\xfad\xd7\'\x1f\xff\xb6\xeb\x93\x8f\xf0\x8c\x9fF\x02\x06\x86\x855z\x94\xd3\x07\xea\x85nd\xd2\xa2\x1c\x10\x1ah\x8b\xa6@\xc7\xc0\xbdK\x02K\xf5\xa3\x84\x81\xb30UN\xc2\x1b\x19 \xb6"\xb0\x12i\t\xee\xea\xfd\xc6\xd5k\xc2\xf9\'=\xc78\xe3\xc2\xab\xca\xe6[0\x02A\xb7\x80#\xb86\xb7)\xa8\x88\x03X\xcf\xe7\xab7\xd4i^("\xc6\xd3\x11]C!`\xd0\x81$\x97<&M\xfc?"\xba\x01\xc6r\xce\x06ZV\xac,<\xdb\x05\xb0\r\xecS\xac"\x91\xec@\xab\xc7swWo\xbdB\x072I4\xd6\xd4xg\x80\xa6P\x17\xbb\x8bJ\xa2\x1aR\x84\r\xce\xcd\xd6^(=Qx\xb4\x81\x85\xb2@U\x87A\xa6E5Wx,\x8e_H\x0b\xff$\xe8C&X\x0e\x88F=J\x84\xb6\x1e3\xa5\x8c[\x80\xd0B1\x97\x1cD\x1f\x00\xf2\xe6\xecS@)\x133\xbbT\x91\xf4_\xdd\xc4\x93A\xe3\x90\xca\xdb\xeb\xb5+\x8b\xecG\x82\xd5To+\x15\x02\xa6$\xaf\xbb\x81j\xec\xa8\x06\xf4X\x88&\x9b\x10\x9alb\x14\x03\xa4q\xfcg\x17\x98\x10\x83\xd6\xa9\xe3\xbb\x7fwB:\xef\xf6\x82\xb6\x85R\x06O\x14OW\xc7\xf3-e\x95\xa4\xa8\xfe\x97a+\xb7\xb3\xfd\x8b\xcf\x94\xf7\x856\xc2\x06\xe3\x9e\\\xa6\x85\xbcM\'\x0e[}J\xd6\xf1\x1e(\xf5\x9d\xe4\xee}w\xf3\x06[I\xb0\xb2\xfa\xd4\x03\xd0\x10\xeb\x9b\xcb\xf5G\xdfys\xb0\xe3\xd9]T\x1e}\xf2\x9d\xc3\x92G\xf9x\xed\x8c\xec6F#\xc9\x0cpP\xa24R\xb0\xe0cdo\xc6.\x80\xfa\xf2%\xba"\xad"iU9\xc0\x8d\x83\xaf\xee\xa0\x9f\xd1O\x93\xf0\x1a\x87-\xe07\x7f\xc5\t\x872\x89]\xfd\x89t\xe1w\t\x074[\xd8$\xc7L\xc3\xbasN\xbe\xe8P@TL\x80\xd1\xe2H\x9b7\x12\xb0\xcd\x97\x1c\xcb\xc5o\xb3\xd9\xc1\x9d\xed\xbb\xfa\xdb%\x18\xedC;\xe3\x9d\xed\xd9L\xc1?\x9c\xf7\xf3\x94\x85\n4\xfc\x94\x1cH\x07\x0e\x1fC\x8f\x17\x1e~\xbb\x8d\xa0\xf7il\x8c\x7f\x17\xe9\nY\xf7\xb1\x9cU\xe25\xa3k\xda\xbf\xee\x08X\xc8%\xb9Zi\x01\xa9\xd1v\x86\x8c\x16\xe2m\n\xa0R-\x80,lxt\x1c\xc0j3:N\xc0k\xe4\x8cgG&|\xd1\xd9E~%~\xc8\xae\xc7\xdd\xba\xf3I\x01\xb2+\x04\x90/\xf6\xf4\x84\xeddd\xc8\xb4\xd4\xf6iV\x1c\xfb7\xf1\xcbt\xa1\xf3c\x1f\xce\xd54\x1f\xfd\xe3\xd3p\xcc\x81\xa7\xd9\xd9\x15>\xcd\xc7[\xa1\xb5\x17\xdd1G\xf3_\x10\x8aZ \x16I\x94\xf0\xda)\xf1\n\x13\xd2\x83]\xf2\x81\x08\x9e\xee\x16\x86\x81FV\x1c\xe5\xdcm\x88(\xa8\x0cr\xee\x16\x8a(Z\x19Vn\x80^\x11\x1a\x8b\x84\xfb\xb7!p\xff\xd1t\xec\xe4\x01\x10\x1bM\x1c\x15B\x06;%t\x18\xca\xb4\x911@\x02\xd27\x1f\xd37j\xa6\xdf\x85\xcc\xf4\xa7>\xb4\xc4\xac"\xfby\x03\xbb\xe1\xf0wjG@\x1bJd\xc1`F\xb7X\xc1\xc3\xfc\'\xadV\xd0\x072*U\xce\xbc\xd7"\x1c\xf5\x1b\x18\xc5\xb1\x11\x00\xfc5Gew\xb3\xf9\xd5\x86lL\x0e8$\xf7x\x87\x82\xa4\xb3#\x04\x94#t\xf6\x1e\xa4\xd7\xfd\xeb\xa3!\x12\x14c\xdd\xadT\x01o\xb00q\'\xd0\xb5\xb7\xec\x94\xf2\xd9=\xa9!\x134\x93T\xab\xe5I\x10\x8f#`b}\x03\xbc\xab\xf8\xf0\xb7\xfc\xcc{_\xe0\x9b\xbe\xfc\x9dB\x08J\x04\x87\x9e}"\x9f\xa1\xc6\x90\x93!S\xb9\xd3\x9d\xdeNwv\x85\xf3\xdc>+\xc9\xde\xf3\xe6]\xe6\x19:[\xd0\xbe7\xf2\xce\xadF\xde#\x02\x1d\xdb\x1f\xfe\xa3\xc0\xf0a2\x89\x0f\xca-yWD\x98h\x0e%\x16\xa5\xec;\x8eP\x8bgi\xa2"8\x14\xcdR\x14l\xa5\x987\xf3G\xa1\xbb=\xb4\xf5\xbcR\xe6\x1eG\xc8\xe5\x14B9J\xb0\xea\x04\xdfJ\xe5A\xdbjPt\xc1~S\xe7\x1b\x821\xe6%D\xb0\xfb6\x82\x8ec\x0c\xe7\x95\xd0\xb4\xf4x\x9b\x8c5R\x14\xccR\t\xf5!\x82\xe2C\x8e\x8c\xc3^8\xf1\x1d\xe4\x10\xc6\x08\xf6\xb2Pf\xee\\\xaf\xbc~\xc1\xca\x01k\x03\x8d\xf1\xef\xab\x17\xef\xb1C\n\xfd\xac\x1c\x0c\x87_+\x83$\xe0\x04\xde\xcd\xa79~\x85\xff\xdd\x8b\xea\x07\xa8(fI\x06]\xfa\xd1\x1d\x9e\xd6\xc2Y\x05\x1b\xd4`\x90\x19F\xd6\xc2h\x04\xbdf\x92GBX\x9b\xc2\xbb\x80\xc3\xb1\xba\xae\'\x1c\xb8\x1b\xcf\xc1\x1ad\'\x03\xd8"\xa0\xa01\xf8\xecW@(up\x84Ro;v\x0e\xc3\x86I+\xeaE$\xdb\x8ch\xa9\x0c4\xd8F\x01\xbd\x18\xbbgH\xfd\x87\x8d\xd2\xde\x8b\xe1\xa3O`\x07)\x16J@\xa0/\xef\xeeU\xe3\x13\x03\xa0a\x8d\xd1\x9d[\xad\xbc\xbe\x87\xb6\xde\xdc|e\xf3\x1a#\x10\x9d\x1b2\xec\x18\xf1\xe3)$\xc8\xac\x05\xfc@\xf3\xc1\r\xe58\xa8C\x0e\xc8\x11\x81\x909\x8c*sH\xb9v\xe6v\xf5\xca\xb4\x17\xf0\\Z\xe0\xd4\x9b\xc6\xc4Z\xe3\xd6d\x0bT\x04\xa2~\xe4\xf7\xe2\xd0\x1e\xc5\xf8NEb\xbc\xd2D\xe0EXZ\xfd\xab7\xe8&a7=\xac\r\xdd\x0e\xcd+\xeb9U*\x9aBM\xed\x141\xe5\xc3\xbc\xe5\x82\x1c`A\xa7\n\x19;i\x97\xc0h.;^r\x13\x9f\xa3\xbcP\x11\x7ff\x17\x98QY{\x14\xb0\x80\xd4\x98\xa0\x9d\xd6\x17o\xfe\xbc1\r\x8a\xac\xd1i4\x16f\xc0>\xfdy\xe3\\p\xfd\x82\xec\x12\x12\x06P_\xd1\x1c\xe2\x8f1\x8aU\nj\x05h\x8aV<mc\x949\x13-F~/\xa6\xfa\x8b\xf3A\xf4/\xa9\x0fb\x7f\x00T\x89\x93\x1eD\x869\xe0t\xc3\xeb\xbd\x07\x0e\x1e:\xd2\xb3wO_Ol,\xb8\xd5\xe92f3\xe59\xb3\x0c\xc3E"\xb5K\xee8\xf9\x9es\x86\xd9\xef\xa0\xd9BBK a\xe6\x9e\xc4\x00\xdaR\xe8\x14\x9au\x1f}\x87\x89\x86\xb0\xeb\xfe\x04\x9c\xc6\xf8\x04XQ\x01\x9a`\x06oA\n\xb8:\nA\x8au\x87P\xbe\x1f+\x01\x14\xaaH\x06G3\x90hD^\x99\xbbq\x05v\xa9\xb2~V\x87\xd7\x90x4\x0e\xfe\x01-N\x87\xc8>\x1a\x1ct\xec\xe7\x8dk\x1cN\x10\xe1\xec\xd1\x00Pc\xc8uDw8o\xcc\x8fl\xcd\xcfl\x03\x9d\xc1\xa0\x869`"\xf7\x8bL\xd8!\xcb\xe7\xdcb\xfb\xb0\x90w\x1ct\xdb1\xd6k\xd7n\x03;\x03\xb5\xcb\xb4\x93\xef\xd1\xb3\xc0\x9em\xcdl\x15r\xe7\xef\xdf\xd6\x1f\x9d\xab>\xba\x1b\xceud\x8e%\x92\x83y\xb0.\xa2\x9e\xe2yD\xf2\xba\xa6|6\x85\xdc\xfd\x1c\x18\x0bRUY\xa4\xc8\xe5\xf0\xcc\xc8H\xc3\\\xf7\xea\xeaIr^\xea\x1e\x89\x11\n\x92aj\x11\xe5p\x880\xbcL*R\xdeY\x957\xe7\xf9{t\x18\xc8\x0eD-\xe3\x04rS+%;l\xad\xbfP\xd1V\n\xbf\xa6$\xf8m@\x9f\x1e\xfe\x0b\xf4om\xb4\xc0~\xec\x0e\xec\xc7X\x13\x9bs\\\x98}\x11\x82\xaa\xa5d\xcb\x17\xd1H\'Y\'\x12\x0e\x04\xacN\x93\x8c,\x99r\x93*\x9b\xd7%\xf7\xeb\x02\xaf\x13#\x03\xe2\xef.\xa3q\xeb\xac\xbb>W[|\x8d!\xf9\xd59`\r\xfd\xb4\x0c\xee\x18\xf23\xaf\t\x1d%\xa7\xda\xc4\x07L\x1fAP-J\x92\x00\xf6\xd1\xf9m\xb7f\xaf\x8ba\xf0\x7f}\xa5|A\xba\xe6\xb4\x80\x0e\xe8\x07\x94\xf1\xc5\x19\x87\xbec-\x95\xc7\x0c\xf4|\x89\xa3Wq5\x92J\xe7\xd2!W\x92\xa3\xbax\x9a]w\xee\xc4R\xf5\xd1\x1d\xf6\xc8\xc8u\x05\x81r\xf2\xe5b\xd2\x92\x1e#oq\xb1\xf05h\xda\x8ai\x0c\xa1\xe8\x10\x015T\xd6(\xa9\x862\x00`\xafP\x97Jy \x8b\xb0\x9a\x06o\xfd\xe5\xaa\xbb\xf95\x07\x980\x9e@n"^A+`\x85\x18f\x98\xe3\xb6\x93\xc0Y\xa3\x94BM\x8a\xda\xb9MD\xc2\xf4c\x0c\xa7\x8dj\xbbf|`t\x92\x10\xf4\'b\xa2D\xc5\x91\xc6"\xe1\x8b\xdd\x93LZ\x85\x92\x14\x17\x9c\x89\xc0\xfa\xa6\xc3\xae\x7f\xb0]\xf2D\x9fB\xc2\x01q\xa25NF\xb2\xc8\x82\x91c\xa1\xb6\x85\t\xc6B\x14z\xaf\xa1\xe0$\xf7\x11n\xe7\xdaZ\xfd\xfe\x8c\xd0\x98(\x16\xbc=L8\xe54(\xc8\xf1L~\x18s\xc6YE\x89\x17\xc0x\x05v\x8d\x7fY\x90\xff\xb5\xf8\x8fa\xab\xbf@\x7f\xf4g\x0b\x91X0CNG\x91L\x17\x90\x88\x12\x13\xb5\xc4W\x06i\xa1\x84\x812\x8c\xb5&%||n\xf4\xees\x88\x9b\x9a\xf5Z\x8dJ\xc8\x8d\x18\xd8(w\xfalcr\x96\xf5o\xca\x99\xa6|\':K`\xd0V\x18\xc29\x13@\x9e\xdd\x98]\x15\xed\xfc\x18h@\xa3\x88V\xe4\x9d/\x8c\x10)c\xf6<\xe8\x1f\xde\xbe\x16\x8a\xf6\x10.\x8d\xe40,\xb0\x08\xca(\xa6\x08a\x04Az|)\xcf\xd1[\x8c\xfbx\xcaG\xefw\xa7\xea\xdf\xbf\x01\xea\xf4\xce\xc3\x1f&0\xd0\xb1\xb1\x06\xeb\xa9?\x9d\xaa.,\xbaw7\xd9\xaa\xe0\x83T9\x7f\xbdD\xe70\xa8S\x16\xa2\x9b\xa5I\xb7\xb1\xf7\xd0\xe7\xfb\xff\xcd\xf8P\x1c\x19\x11\xf8+\x8dvG\xa2\x89!ZxUGC\x89\xaa\xd5\x9e\x0b\xe7+E\x18\x88\x9e\x83\x08\x02\xf1\xc654\x1eb@IS\xcbUR\xcb\x85\x8d&o\xeb6V\x19\x07c\x0e\x14\xbfx\xf6$\xcc\x13\xe5\x0fN\xf7\xd1b\x19L\x13\x12\x95\x89\xfcI\xfa\xd8b\x9b\x0f\x03\xd1YEQ6a\x9dB\xe5\x9b\x81\xef\x1f)Y\x8e\x17\xce\x05\x1a\xa0\xd8\x02\xaa\x03\xb0\xc7\xb9\x12\xfa\xc9S\x06\xa8\xeb9\x07\x05\x81\xb6\xd9\xec\xf1\x16\x04K\x11\x1fw\xf2\x07w\xe5\x15\x87\xea\x1a\xb7g\xdd\xbb\x0b\x9c\r\xed?g\x9a\xe41\xd5\x05\xc5\x93@\x88]Q\xde\x896}\xe9-\t7\x97\xe3\xc4*N\xf6,\xa6\x02\x92\xc8\xab`R\xfc\xe7c\xbb\xc6\xf8"\xb2\x1d\xc3\xcf1\xa1\xe5\xc5\xea\xfa\x826\x0c\xb3\\+\xc0Y\xb5\x11\\\x17\xa6\xde\xa8\x81\x02~\r\xe9|\xd1\xf7\x179I\xaa\x0bMR\x06\xec\xfc\x81"\x96\x99P=\x14\x060@U\xe4H\xc5\xdf\xacb\x9e\xb5\x836\xd8\xb0\x8c}\x92\xb7X)\x02\xd2\x8e\x00\x95!\xde< sb\xed\xd9\x1d\xd0\xc6\xd16\xeaP\xb6\xd1\x05\x0c\x85\x87i\x16\x06g\xf1o\x8d\x14\xd4\xf7\x8e\xcb%\x9f8\x9e\xd6ry\xe3\x9e\xce\xa4\xf3\xe7\x18)\x88\x12\x9b\xa8\xb4\x05\x92B<!\x9a\xb5R\xb6\xc9\x11sV`\xd9,\xc0 \'\xc3[\xc8\x98Ik0\x9f\xc1\xf3*\xbe\x83\xb5oI\x83\x132\x91f\x82\xcdn\xf6\x15\xf8\x17\xcb2\x8c\x8d\xc6\xd6>\r\xedP\ns\x06\x10d\xf2\x0b\xce#\xa6/v\xfa\xbe\xe0\x84a\xfeB\xb8\x0ft\xc7\x81\x82z\xa7Q]y\xae@\xa7O\x81\x9c\xe5p\x0b\xbe\x0f\xd4J\xc3\x83@LQ\x9fz\xea>\xfe\xba\xb14^\xbf?\x81A]\xff\xe9\x03f;\xa6 \xad<\xd7\x93\xa4\xc1\xe2\x12\x08\x9b[\xad\xbcYB\xfeY<\xe3\x9e\xbf\xa5\xac\xc7\xc8V\x08\xf9\xa0\t\x13\x1f\x84\xa1\xa0\xb3K\xbaPx\x07\x9e\xdd\xaa^\x9fw\xe7\xa7\xeb\xcb\xd3\x02+\x9d]\x86;\xee\x05\xd8xRT\x87\x89\xfd\x04\x91\x02k\x05\x00\x80a\xbf\x10\xfe\xb6\x00 \x9e2O\x94\xbcG8\xfe\x82\xc0}`D\x83+\x00\x8a\xd0S\xb8)\\\xdd\x01\x86\xbd\xd4\xa13f\xbf\x95ic:%\x1d\x1f\x95v\xbbde\x9dhl\xb7wh+W\xc3qr5\x08{\xba\xd9\xc1\xf0k\xc7\xf3-\x18\xbf\x96\xa3K+\x06\xbd\x0b\xb1\x13Ae\xc9\xb3\xde\xf1\x1f\x86D\r\xefY\xe8\xe9\xc8\xefGi\xbc10\xbb\x85\xb3\x84s!\x84Ce\xed\xbc{\xfa)\xb0Ic\x8aT#\xf1\xcb\xb10\x97\x95^U 3\xf2\xcb\x0e\xb9\x1e\xd9\x81\'U\x01.vA~\xd7\x8e\x1f\x0b\x1dC\xa0%\x15@\xb1N\x9e$q\xc1f\x0eG)Uj\x199\xf5D\xc9\x0b\xc9hV\x91\x14\xc1\x82\x1a\x07,\xede\xfb\xd3\x9bH\xbf\xaf/\x84\x9a;l\xe8\xa8d\xc7&3G\xec\xb9\xb7m\xf9\xa2=`c\xf9\x810#\xe4Oc\xc1-\x90/z\xba;\x17\x7f\xc9\xc7\xa1\x8al\xb6\xb0\x8b\xd4\xd5l~H\xea\xafY\xd6[u\x96\xf4\xcc\x9fi\x95G9+\xd3\xbd\xae\x8e\xca)\xc6\xdeGm\x1a\xdc\x99P\x95\ra\xdaRg\x17\xa8K-\x80\x1f\xfbG5\x97P\x8d@N\x16\xd0\tZ(\xbd]\x1d\xa8\xf5\x92\xc2\xf7\x1b\xa3\xcb{\r\xe6-9\xfa{\x9d\xe1\xefm\xebL\x0fqQ\x90\xeb>\xfcH\xf7\x0f-\x01\t\x1b\xfb\x80U\xa2\xa1\x81\xcc\x0b\xf9\x1c\xe2)\x18\xe3\x941\x03\xed\x98l\x9a!\xf4\xeceN\x8ck{\xab\x9f\xba\x12(:v\xe5h~\xf9\xb6\xdb\'\'\xdf5\x93(u\x89\x07\x1e\xb4\x9a\xb5\xf3\x84_\x8cPqR\xb0\x86\xc7@\xe7\x0f\xc6\x02\x8a\xf9\xf2\x00\xe7\x80\'\xa9\xbc\x8a\xbe\xf4l\xcfPY\xc1g\x97.+<\xebj\xe2\'\xf8V\xf8BU\xe6\xc6{\x8b\x08qR\xfcW\x89\x88a\x93EC\xb6\xb0\x93\xfe\x9b\x06\x9d\x87\xfe\xc8\x0f\xb0\xcd\x9b\xdde\xea"\x83\x11\xf0\x7fQd\x88\r\xfe\xffOd\xecR"\xe3=\xc5@X\xc8R\x06\x0e\xdf)\x07ByG\xe3\x9a\x10~i\xd2\x8d\xf7p\x0c\xcf4\x86\xac";\xd2\xd83%*HD\xca)\x7fl\xf7\x85\x01\xd9\x90\x88\xef\x00\x83]\xb8\xa9\xa4\x07\x9e\xf3\xbe\x85"x\xf1\x1e\x1a\xefX\xe5\xc9\x9fI\xd7c\x9bA\xea\xc5\xb2(Sw\x13\xb6\x19\nTZ`Br\xccqBBTdO\xfb\x8a~#\x9d\x1fS&5\x95\xf9RFfD\xe8fQ\x95^\xfd\xde?\xf0*\x89\xf1\xfd\xdfR\xc6\xb6\xd9\xfcz&\xe4\xe5\xdf5\xbf|"\x10\xc7<\x84\t\x9d"\x8e*J\xa5\x8b\x9c\x9f\xa8\x97\x8c\xa1\xce\x93U\xc2\t\xe3\x15\x80U\x10?Zm\x1b+\xe2*\\\x13\x14?\x9a\xe9\x81\x03\xb5\xf0\xb5\xaa\xd4\xc8\x9c\x97\xf9/\x1cx\xe8\xa8\x94uv\xaa\xcc6\xebs\xd6\x89\n\x1a\xaa\xe6V\x05wb\xd7I&\xc2\x0b\xa0\x0cn\xed\xac|O\xcfjk\xef\xb0\xa9\xe9\x90\x04=\x18\xf1\xd4o!7\xe0\xf3\xa9\x04\xdc\xc0\xe8\x03\x06"\x96u\xb5\xd8\xdfa\xf6)\xbb\x84\x7f\x81\x7fU\xf9V\xdf\xcf\x85J\x0e5N?E\xb7\x90(\x9b\x93\xa7\x11\xef\x88\xf2\x15\xf9]g-\x9cf\xc2H\xbf\xb6\xd2\xb8p\xe6\x97;\xc4F\x05O\xfe\xd3\x9d`%\xb38@N\t\xeaf@V7G\xcc\x9a\x9c`\xf7\x17\x98\xd2\xd9\xdb"\x96\xeb\xf7\x05\xfeW\xbb\xc2\x02\xcd\x0e\x94!B\xe9\x9f\xc4B\xa5&\x8f\x17\xfb\xba\xd8o\xa4{\xbc\xa8\x8e\x93=^\xffd_\x97\x8a\x1byNG\xaa6,\xca\xd8\xd1\x9e\xc3\xbd\xb2[A3Ai\xdec0\xd8\xe1oz]\xefM\xf0\xff\xd4\xbb%\x1dy\x14{\xe5\x1aJu\x80\xc1\xd1\xa3\x9f[\xfe\xd5\xbd\xfcIX}\xb4\xcc\xc6\xf8\xf7\xe2\xb0\n\x1eS\xdbvSy\x87U\xf8y{\xc4j/\xb2\x81\x9b\xb1\x86L\xb0\xa5S\xc5|\x01K?\xb1\xd4?ib\xb0\x95\x8b\x181Y\xc4A\xcf\x0b\xa5<ke\xf5Ys\x84I\xcb\x12\xa5a\xa0\xe4\x16K\xe5\x02%\xa6s\xe7\x17\xac\x91\xa3:)\x16\xc9\x95\xb5\x99\xea\x0c\x15k.\xac\xa88>Wi\xb9\x9b\x17\xb8\xcaUT\xbbM\x8b\xca\x88\xc6\xf8-*\t\x9c\xc5`\xed\xca+yR\x1f\x843$\xd1{p?V\x9b\x8b\xda\x03\x87j\x13A6\xfcqO_\xcf\xd8\x87\xf9~L8I\xd8\xb94\xaa5%;k\x01+t\x7f\xd2\x11\x8bc\x1f\xa7h,\xa4L\x0b\x17\x88\xcd\x1a\x0e\x1f3:\r\x0ea\xfb\x82\xe9"\x9d]\xa6 \xb3\x96\xceU.\xb5+/PH\xcb\xc4aQ\xffq\xf5\x0c\x07\xc99\xabG\x0c\xec/%\xf0\xc7\xcb\x9b\xb3\x88E\x9en\x9b\xca\xd3\r\x03\\B\xc4\x9em\x8b\x95\x18\xaf\xaa\xc9q\xcaYd\x7f\xb9\xc3\x06l\xa0\t?6\xf1(\x12\xb5\x90\x05\x14$\x98C-\xe2\xfb\x0bS\xee\xa3o\xd5z\x986\xf9,U\xc5\x12\xee\xc4\x14&\xff\x8b\xaa\x92q\xcc|_:\xe7^\xc3\xe2\n\xb1\xec\xb5I\xd8\xe8\xed\xaeT\x14\xb4\x07\xb3\x8f\xe5zQ+\xe4\xb3\x08\xb8Bd\x08\xfav\x87"\xb0\x80\x15l0B.sY\x8c\x8fG\xb2\xcc\xef\x94)F@\xa3g\x1e\xb8\x93\xd3\xaa\x8f\x10\x87\xfda\xa9L\x9d\x8c\x85\x9f7\xae\xb9o\'\xb1\xc4f\xf14\x1c\x9c\xb0\xcf\xd5[k\xee\xdb\xaf\xbc\xe1\xbc$\x88\xe6%6\xe7\x94\xfa\x12I\xdb\x82\x89\xa4m\x86\xf7\x9b\xa6\xb7c\xcdxP)\xa2p\xc0\x9b\x19Jz\xe871\xbc\x9b\xf7\xd6-S)\x1d\xd9\xd5I\xfd\x88,>\xb5\xdd\x17\xbf\xc1\xa2\x91\xc5e\xf4\x8c\xd2\xb1\x06+\x16\xd9 \xb4b\x85\x1eN\xf7\xf2({\x8b-n\xce}u|\x1fdRk`\xd5\xea\x1d\xb9dY\x07\xaa\xbad\x90\x9a\xcc\x9eg\xd0iE\xb1\x16[\x01\x0e\x9f\x9a\xcd\x1a"Ka\xd1\xee\x07\xe4\x93\xe8\x17A\xd2V(Y\x9c\xc2F\xd1wQ\x9d\xc8\\L\xca"\x07X\xf5\xc6\x1f~\xd3A\xea\xb5d\xd0\x86\x15T\xc6v\x87x\x17I\xaa\x0c\x99v\xc6\xc4\x1a:\xb5@\x9fy\xc3\x12\xdd\xab<\x91M\x0b\x88\xc6S\x16\xd5\\R\r\x9a\x97E\xc6\xb9;\x98C\xc6\x85gZ\xd6#\xef\x9c{\xfb\xa1\xd0U\xee\xcf\xc1\x0bM>\x02\xc4hBL\xdcm(a{\xbcE&\xf9\t.\x0bg\x05\x8d\xda\xa2\xf1\x03\xb9 ~\x16\\?\xab\xf8\xb8$\xdeP\x9a\xb5\xcd(d\xca\x9c$\xa9m.\x95\x9b\x01j\xf7\x7f\xb6\xbf\xeb\x0b\xdd\x02\x81\xfd\x93\xbeS\xaeU\x12\x81\xf0\x1f\x7f\xe4\xc2c\xa4\xe7\xa6f-\x8d\xd3g\x04\x91k%\xcaA\x1cX\xd4\xea\xc0B\xcb:\xea3\xcc\x02\xa6\x94\xafK\x00o5\xbb\xcf\xa3Q\xfd\xbd6\xff\x8b\x19\xef5\xff\xe0\xb1Xh\xde\xa4\x89^n\xad\xdc\x92\x98\x9b(\x80\xb8(\x0b\xa2\x9b\xf5\xe5&\x13D \xea\xfe\xe9\xea\x8d%\xe6e)\xb1E\xd2@c|\xb1\xb2~\xad\xfa\xe3\x99\xca\x9bY\xdd\xe8\xc0\x1a+\xea\xe7\xe37:Br\xea\xc0Z\xcfPN\xa3\x8fn\x90\x05\xd57\x12\x9bd\x8c\x84\x12\x91Q\xdbXw\x1f\x7fm\x8c\xcaW\xc7\x0c\xc5a\x91`A\xb8\xc8\xb6\x95\xac \xda<\x10>D~\xa9C\xc5A\xa6\x8c\r\n\xe3\x91\xfa1\xc8"q\x16\xfc\xd44\x82Q\xc2\xf8p\xe7V\xab\x97\x9e\xb9\x93\x93\x81l\x8ap~o\x91\x16\xac\xb8[\xf6\x15\x10\xa9\xbe\xca\xa6&\xa7\x98\xe4w\xc6\x98\xce\xc1\x0c\x9bbY\xa0mdY\x8a\x02(\xde\r\xee\x08\xc5"\xb7\xe2\xd9\xa6\xc4\xb4f\xbe\xf5\xaf\xe2\x00\xd8\xbbBQ\xe7&P\xf9\xb4\x87R6V|\xb0s_"\x85@a\xeb]=S]\x9de\xfa\xf2\x9a?\x05\x01\'Jq\xa8\xa0\xb2\x14\xf5V"\x9b\xf30\xd6\x18\xc6\x98\xf1\xef\xa1o\xf9\xa5M8\x1f\r\x81\xe2J\xb9\x82DC\xa9\xbcH\xcf*`\xc7\xcc`\xd7\x12\x8fx\xf4%\x12#\x89\x8d97\x8d\xfd\xa0\x88n\x9b)\x85\x8a\x9c\x9fp\xfe\xf2\x96,\xa4\xf7\x86\xe2P\x99B\x07\xb0\x90\x99\x1b\x89R\x8d!i\xd7\x0e*u\xd1\xf0\x08tD\x84\xdb\x84\x7f\xca\x1b\xc6\x97\xb7\xda\x9a\x1c$\x07z\x83\x1bj%\xf5\x873\xa8\x9b/<\xf6\xf5\x98\x11\xf9\xa8\xc1f\r\x94{\x9e\xc9\x00\x95\xab\xf8\x18\xa3\xddcN\xec*\xe9\xe1\x9b\x83\xe0\xc2\n\xcc\xf1\x19\xed\xe3Q\xe5\xff\x9e|\xe9\x9e^\x0c\xf4\xbbbo\xb0\xd7\xbeUK\x92\xdf:\xe0-\xc9*AS\x92U\xe8\x8b\x88\x84\x840\xc6P\x8ak\xcd\xb4H\x84#1\xc6\xa4w\xcbs\x9f{\xdes\xffDq3\x95\x8ajNW\xfe\xa9p\t)X\x02\xc0\xb5\xd3,\x8a\xabc:\x01!\r\xfb~\x8eBVe)\xe9\xe4\xf9\xe0\x9b\xc6\xc3\xcb\xb2\xfcAe$\xfb~\x8b\xf9\xc8-\xb6;hx\x98$\xfe\xd9\xfc4d\x19jr\x04,\xc6\xdcIG\xbaR\x94\xffN\xec\xabP8\xcf\x8dc\xaa\xb3j\xac\x02\x06o\xed%\xa6DV6\xcfV\xd7\xe7u\xcd\xcbs\xd3I\x17\xad\xb2e\xf9\x00\x94\xbd\rU8:\x90\x9e\x0fB?\'*fU\x17,\xe9\xe0)\xaaTh\x1a\xac\xdds\xa1\xd0"\xf4\xa2\x0c\x19oA\x8e\xc0^ \xd4`\xc3\xbd0\xe9\xce?ds\x9d\x1b\xe5z2\x9b\x0c\x86\xb0\xf0\xcb\x90\x99)\x13\xdc8\xa5g\xb2\xc7\xe9\xb9\xb7\x82\xa0\x99O.\xda\xd2p\xbe\xdd\xe2\x03_X\xe6\x988#7\x9b\xb7"\xc9N\x08\xb6s\x94\x93\x81\x1b%][t\')\x17\x81\x14w,\xcc\xe7\xcc\xef\xd7\xdf\xd7o/\x07S\xbdiSZy\x1d\x80\xd6\xb52\x13\x82]T\x96\xa8\xa4\x10zH\xcd\x81\xba\xb45\x05\x07R\xeeS;K\x82\xa0deF\xd8q\x8a{@\xa5\xe3^\x1ab\x98\xf3T8P\x11\xdc\xe7\xf3\xb0\x11z\x12\xbeNO\xa8\xd8\x93\x1bU\x94\x98O\xbe\xc4\x86^\xd4DI_d\x18|\xd2\xb5Z*\xf2\xa2\x8ew\x9c\x88)\xcaCv\x13\t\x8d\xca1i\x84B#\xdf\x92c\x8c\xe9\xcaM\x16OT\xa6SU\x06\xcdt\x1a\xe2\x84\x10\xa4O\xc9z\xca\x06p\xa4;B\x19\x00\xde\x8e\xca\xc9Y\xe3g\x92\xc6d\xcdq\xaf>\xda\xf0w\x10~oV\x13J\x8f\xedHmG*?\x12\x08\xb4\xed\xb2&\x97lQr\xae\xe2D-\xf7\xb2Y\x01r\'\'\xea\xcbOD\xbf\xdd\xc9{[\xa9\x12b\x06]\xffa&\xd3\xdc}\'\xde\xa5\xf6\xf4\xe6\x92\x99r\xcaR"\xb8\xcd\xd3\xd4\xa8\xc4`\x04\x0b\xde\xdbe\x92\xb41h\xa7@\xee\xf9\xb5\x84\xc7S\xac\xb2aj\x16\xabl\xd3S\xb5\xa7\xafk\xcf\x9f4\xae>\xa9\x7f;\'\xd60\xb9\xdc\xf8j\xb9\xf6l\x16W\xa5z\xe6\xf9T\x04\x89&V\x8cb\xf1r\x0e\x00\x89\xfe&\x8a\x9fx\xb1\xb4\x18\xd0S\xca\x05\xa1\x1d\xe1\xbe\xd1g6F5\xe5IW\x91"\x0c6X\xd9\x81\xd5\xef\xb7\xb1\t\x91\xe7\xe6+Z\xd9\xfc\x10\xb6\xfc\xc0\xdc_\xdc\'\n\x08\x99\xdc\xaeAm-\xb6\xe4v\x94\xb9Q=\xb7\x89\t7$Z\xeaw\x1e\xa2\x03\xf0\x9c\xd6\x1d\xe5\x87\xd7\x8d\xc5\xbb\x18\xb5\xbd\xfc\n\xb7Q\xb6\xc8\x0e.\xbe\x9c;\x99Cw\x15\xaf\xdd/,cp"J\xe4\xf8\xe1\xc74\xe6a#5\x02\xa0\xdaIi\x1a\xb0F\xe7d\xf2 0\x83\xad\x10\xa9F&\xa0\xd6\x8a\xc2\x98+/\xf4^\x87\xbaf\xe4\xce\\\xa9N\xce\xf06\xca\x8c}\xa4R\xee(\xe0>\xb9\xd2X\x98\xf1\xa2Ns\xdfT\x7fx\x13\x12\nk\xa6MRQZk\xecD\x80A\x85Q\xe2\xa1e0\r7\x90mh\xce\xf9\xf7\xf2,\xf5"<B\x8e\xd7\xe0B \xdf\xb1J\xbeC\xa4v\xe5^c\xe2"7\xe8\xc1\\\xca\x8e\x9f\xdf\xdc\xf8\x9d\xe1K\xb9\xa3\xd5\xca&\xc2o\xe0\x07|2\xb2\xa4\xdd\xda\x95-gm\x07e\xace\xa2\xe5X\x84\xb3\x18H\xcbE\xffs\xf4\x93\xd8\x98~.\xe9\rM\xfd\x08\xd2ez\xf3\\!\x89%\xcds\xedl5\x97l\x89\xfa\xde3\xb6N0\xd9\xde\xec\xbf`\xde\x90\xe0|\xf8\\~\xd1`\xda\x19.\xce\xe4\xf39pL\x91\xe0\x075\x1e~[\xf4z\xda(V\n\x9e3H\x1eggA<\xe8\xd6\xb7\xf0\xbfi\xd1\xce\xca\xda},\x98\x01q\x12*\xf15\x1dX,\x13\x8e\xe3_5\x1d\xc7\xd1\xd1f^\x1b\x8b\x19z\xe9\x844v\x94^,\xc6\x8b\x8d\xfdJv\xd3\xb4s\xa5\x90\xae\xf4\xc8\x93\x88\n4`\xa4\x03\x06\xbd\x189\x07%\x8b\xacH#\xb7#g$\xd7\xc7\xbf\xc5F\x8f2\xb6\xcb\xea\x08w\xa9\x11\xfd\xec\xa7\xa7\xb8\x9bOcj.\xd8\xcdf\x14-\x19\xef\x82\x81\x7f7F\xa9\xd8k\xec?\xbe\x1d\xe5"/z\xe6\xef\x07\x85O\xa8e\xc0\x18w\x9a\xc3\xcfM\xedZ>\x14\xedZ\xe0K2L\xb8\x19\xb81\xaa\xf7\xcc\x88`\x05h\xd9\x19\x941I\xca\xdc\x00ul \xe7u\xce\xf9\xb3\xd5\xdf\xc7\xb9\xdd\xc9\x8cM.\x02\xecNc\x97F\xd83\rT\xe2\xeb\x1c\x8dM\xc9\xeez\x1d\x04)\x9c\xaa\xf2\x99\xb4\xc1\xdc\x95;\xd5\xe9\x17\xb5\x87\xabu\xd0\xa6\xa6\xef\xd5\xeel(\xb73M\xc3Y0\xc1p0\x83x\xb4\x88\x8eqo,\xe1s\xb53\x08\x95,.\xc76\xc3(\xfd\x14U\x8b\xfe\x87\xb0q\xe330c\xfd\xed\x9b\xfa\xab\xbb:Ds\xab\xe8 \x1f\xff\xc1\xd7$\x9a\xb2LE\xa7)\xea\xc5$\xc1\x1cv\x82\xed\r)T(\x08\x060W\xb6\xa8\xdbS\x1e\xc0\xc3.\x8dB\xcf\x15\x0c\x07\xea\xc6@\x11\xa01`;\x1dN&Ez\xa2"Xq\xb9\x05y\xebXiW\xd1\xb5\xea\xc6\\\xf5\xe29w\xfazu\xe9fc\xfc\'w\xfd~e}F\x94\xc8"\x14\xa5\xe2H\xb0\xa2]\x8609\x8c\x90\xca\'\xcbhmh\xd6\xbc\x02\x85\x9b\xcb\x82Rl\x99Yfs.\xa7\xf7\xec\xf5\x95\xef\xaa\x0b\xaf4\x94\xe9 T\xd6~\x02\x04\x05\xf9\x99\xb0\xa4\xaeK\x89\')6\x9f\xf0L\x1a\xe0\x80ag\xf7\x87\x1fvv\xfd6\xde\x01\xff\xef\xdc=z\xf8\xd0\x11\xa0\xdea\xe7\x7f0!\xf4\xa6\xbaG\x15I\x8ci\xc1\xbf\x8f\x02\x9a\xcegXwO\x05\x08 u\xcb\xd4\xf1\xcfL9\xd8\xa9J\xb4\xb8$\xceF\xafk\xc6\xe2\xbe\xf6E\x8b\x0b\\\xe9\xe0\x1c(g\xbc:u.\xc7G\x93\n(\xf9\xd19ws\x128\xbc\xb1p\t\x15\x03\x0c\x8c\xddd\xfdN\x9c\x81\xb2\x8b\'7<\xad\xcd\x8c\x83.\xd1\x8c\x888\xd5\xb0\x12\xec\xd1.\x80\xdd:E\x05m\xd1C}=\xc5b\xbe\xd8\xa6\xa1Ia\xb8\x87\xde\xc1\xe8{\xf3\xb6\x82\x19^\xd6;\x89y\xfd\xa2H\x05R\xdbj;z\x9d\xbe\xd0h\xa9s\x14\xfc\x8b\xcb\xa2MdB\xe3\x8e\x9f\xdc\x13@\xf1\x80\x88\xe1RW2}Y>\x80>7\x8b\'\x05\x05\x95\x0b\xe8Q\x00S\x16\'\xf6\xe2!\xdc}\xb8d\xe2\xb9\x963\xac\x8cY\x00\x8d\xa5\x1d1\x82%j\x85\x8c)p\x0f\xe7\x04U_#Q\xb1\xdb\x02\x1b\x85]~\x85\xe2da\xa5\xb26\xce\x00\xbb\xd3\xdf\xb2\xa4\xad\xbd\xb8\x04*\x0c\xefO(\xf91\x93\xfa\xbd\x080\x9d\x8d\xd8\xc2\xf6\x7f$D\xb1\x0b]\x12\xfd\xee\x94i\x91\xc9`73l\xac*#1\x0f\x1eT\xbf\xbb\x08\xd2\xad\xf2z\xbc\xb1\xbe!n+\x81\xe3f\xe9+,[\x9f\xdb\xe4\x86\x9d\xb5\xc5Mw\xfa\xach\x06\xe9+\xdf\'q\x1fa\x8c\xf3\xbb\x92mfY\xf0`t\xf2\xf1\x14/\xa9q\xf5bu\xe59\x1a\x19\xfe\xb5\x85\xc9\xeb>KtwT\x82N\xb0\xba,\x92\xe3\xc4;e\xb4\xc9\xe6\xd80\x19S-\x08\x9d\xca\xfa]N\xae\xf3\x1a3\xfb\x1a`\x03\x1d\x15\xc0\x1e\xb5\xf4\xf0|!\xefh\xf1yQ\x0f\xdcF\xd7*u\x8fz\xb5\xcc\xc2\xa4\x8e(\x16\x86g\x1e;{\xdc\xdc\xd9\xd5\xa1\x1a\xe9Q\xcf@;\xc7\xda\xad\\\x96\xba\xb3\x80d\x08\xf7\xbb\x05\x99\x9a\x01&\xc7\x0f\x8a\xdc\xfb\xcd",Nk\x9e\'\x8al\x88\xb21\xf2;9\xed^\xbb\xe1\xdez"\x8b\xc1\xa8\x8b\xed\x83\xbb\xee\xe3\x8b \xfb1\xa0F\xe4U\xbds]s7\xa2>"\x91\x10\xcf\x9f\x0c2c\x06\x1d\xce&v\xe3\xa6\x1aiO@\n\x90\x8b\xa6\xed\xc8\xd6\x7f\x94\xe0Sl\xc7n\xf3\x86\x85\x8c/\xa8\x9e`b\x81"D\xff\xb7K\x8d\x85s\xacVy}#\xd4\xd8!\xf6\xc7\xb0\xc3J[\x89H\xbe\xa5\xfd\x00(\x13\xb5\xf7\xe2D\xf7\xc4\xb1V";5S\xbd\xbc\xe9\x93\xf5[\xb9\x8f@\xbc%\x11\x0b\xd1\x80PV\x19M^\xf1\xe6\x10\xbb\x1cQ\x02\xa7P\x1cdd!\x05\xa50y\x14(C\xd3\xac^\xf2.\n\x81\xfbtJ\\\x86\xe0\x8b;\x00\x92-\xe3\x08vD\xcfZ$R\xa3j\xcb\xb0\xc7\xfe\xf1\xdd\x1futt\x9cPT\x96\xb5\xa81\x84vWQ\xef><-\x9a\x8ef\x03;A\xf2Q\xcem\x19\xa8F\x91\x9b|3\xd0\xccH\xf0{/\r\xe6\xf1\x94p \xb2\x1eD\xf4\xe8\x11\xd7\xcb\x97Z\xf0\x89gg\x85GA\xcci-\xf2f#\xe4\x1b\x91\xe9\xd3\x87\x81\x03d\x1bIP\xc2\x85D\x1aG\xca\xf82\xdf\x1f\xdf!\xf4R\xba\x80\xc5}\xf9\x13g\xef\x04 a\xadU\xc2@\xe1\x08\x98\x1fq\x17\xcf\xe6\x81\x86\xf29;)\xf3jdS?\x03\xa98\xa3u\xe4\xd4\xb9\x8e\xabo\xe1\xcd\x0c\x0bT:\x0c\x1c2T\x81!9\xaf\x8e\xe4\xb7\xd7\x90S\xb1\x1a\x9e\xab_M\n*_\xba\x89\x11z\xad\r2\x9f\x02\n]8W7\xdd\xb1\x16-\xe5Kf\xa6[\xb4\xcfJYN\xb2;R\xbd\xf8M\xe3\xea\xf9\x9f7\xe6\xbc,\x97\x1b3\x11\xac@\xb4K\xdd\x11\x91\x9a\x01,0d\xe9RT\xd1\xa9i\xa4)3\x1e6\x9a*f\x8bV2_\xe4+\xe7\xb4\x06\xda\x9a\x16A\x1a\x04\x19\x04L\x0b\xd5\xa5\xf1\xdasa\r\xd4\x9e\x8d7.\x9cA\x0b\x08\x14\xcb\x85\r\x916\xa5\xe9\nJ\xc4p\xa3HO\x9d\xe4\xce\xdeE\xab\x80\x88O\xa1\xc2\x98\xe7\x9c\x10\t\x97\xea\xce\xefqh\xd1\xe2hL\x9ad\xb4\xe8 \x89>,\xef\x15>s\x1a\xb7^\xd7^aN\x88h\xbd2q\x11w\xe1\xe6\xbd\xfa\xdf\xef\xe8pK\xe0(\xee-\xe6M\xd0r\x01TP\xd6\x18\xd2}\xe8\xaf\x04qR,\x97\x06\x115\xe4xw\xe8\x1c\xe5(#\xa5aKV\x92YL|\xe4\n\xea\xe1doYCn\x88k\x9a\xa6\x1f3%\x88+\xde\xe8HDM\xe0\xa7o\xb0M\x01\x13\xc6\xe2\xe9\xca\xebY\xafh_i^D\'\x83 \xe68\x0e\x84=\xcf"\x81$(t\x9aaC\xb1\x08\xd3\x89!\x1b\x1eb\xc7\xbd\x88 W\xce!\xa1|\x8b\x88^\x8d\xc7]\xd4"L\xa3\x95\xb5G\xf8;\xeaN\x16\x11a\xac\x1f\xbe\x87I\xe8)v\x16\x8bpLK>\xf5\r\xb4\x8b~5?\xadRV\xe8W\x1f\xd1\xd33?\xa1\xab\x99\x9f\x8a>Z\x7f6\xed\x92\xb8\xf4\xc1\xf3\xd5\x89 \\k\xca%\xb1\xc5\x02\xabza\xa2\xb6~\x81uZO\xb1\x87\xc3\xe7\xf4S\xfd\x1a)\xa6b\xd15\xc6G\r\xcd&\x06]\x1b\x05{\x89\xcc\xc1\xc5\xd9\x08\x9e\xe0\x966\xce\xfa\xe6\x86I|\xe0\xb5Q.\x17\x9f\xfb\xc2\xc1H@\xd5\x17\xaf\xd4\xe6\xcf\xd6\xae=\x83\xadGt\x9c\xbf\x89\xed\xd0A\xeb\xba\xb3.\xeeF\x98^h\x8co\n\xa9\xe5S\xf3\x061\x13\x1b\xd9\xb9\xc5\xa1\xf79\x1cyh\x01\x90t\x1a63\x99v8\xad0\xb9\xca\xcer\x80\x13}\xdd\xf9\xb4\x81y\xa1Y\xd6.8]\xd2#Q\x15\x9a \xa9\xca\xa4\xa9\xabg\xc2j|<\xc5\x1f\x01\xdc\xda\x95\x17\xe2\xde\x87\xa0V\x18\x04O(\xc3!\xf2\x17#\x85(\x9a[\x9d\xe5I\x8b\xdc\x9d\x9a\x14\x10\xac\xc6\x9dL`\xdb[\x06m\xf5\xf4a\x11\x07g/=\xc5\xc1\xc5\xcdYj\\\x0c#r6\x92f4x\xc6g\x18t\x12\x12]\x03o^\x01 ^;)\xc4V\x86*\'\xe8\xbb\xc6\xc8\x891\x08\xbbX\x1a\x1c\tF\xc34$\x88\xdb\x00He\xe1\x8e\xd5\x0c\xfc\xd6\x00o\xa5B\x85\x85\x91\x88\xf0\x05!\x1b\x964\xd5\x84\xd7\x15\x98\xb0P\x08^\xac\xa5\xff\xda\x83\x96ac\xd2F\x9bb\xea\x1c\xdfN\xd0\xdc,&\x0c\n\x8f!\xb7z\xcbO.\x01\x9b?O\xd9\xbe\x98\xd7\x8c)\xa7v3\xac\xc4\xa4\x17\xefU/\xbf`\x99\'\x0e\xee\xe9\x05]8s\xc6o\x10\xdcV\xc0x\xd41\x8c\x9e\xcb\xe4P4\xe6_\x85\xb0\x8fC\xac\xe2\xa3\x8cre\x1cos\xf9{@G\xb2\x91C\x90\xb3\xdb\xd1\xbdG\x86\x03\x1e\x13\xbc\xe9\xd4Y\x86\xaa\xf6\xb1G\xbcw3F\x13&\xf8\xd6\x1a\xd6/\x96n\x02\xb7\xf3\xa1\xc4\xd4\xe5>\xf9\x91\xc9\xaf\xfa\xd5d\xf5\xe9\x04\xcb\x05\x8e\xc8\xb2\x11\xd2\nC\xdbZ\x842\xfc\xa5\xde\x81\x1a\x9f\x92\xf9\xf2R\x0f\xca\x05\xf7\x89X_tU\xa37w\xf3B\xed\xd5\x06H_q\xd1\x1c\x1f\x00J\xe2o\x07\xa4\x82\xe98\xa1[\xf7~\xae\x8dmP/v\x86G\x8b2i\x81\xa6\xa1_Q\xc4d\x8c\xe9\xeb9\x0bMw\xd0\xcf\xe0\xe8O\xd9\x8e\x90\x1ez\xad\x87\xb2p\x84\xdf\xe3jc\xe2\x12\xf3\x1b\xa2\xe3\xda\x8d\xc6\xf88\xea\x1e\x84\x82\xfa\xc6\x03P\x8at\x87`8O\xbe\xd7\x16\xea\x16\x18Z\x8c\xe4\x1d\xd9\xc2\x14S\x00\x93AH44\xedw\xa66\xdbf[\xcd\x1f\xb4\xd5\xde\xf5\xfe>\x9bAL\x97\x8b\x94@\xc6~5\x15Z\xd2\xb1\xac\xaa\xf7\x82c\xb0SG\xdd\xbb\xa2,\xa1\xf7\x85|\xcb\xc3\x84R*\xfeg\xdf\xa1\x83\x06\xda{\xd2Y\x0e\xc0\x91\xd3I\xc9<\xcc\x95\xff\xef|\xed\xb05,\xcb\xe3\xd0\x19\xd8\x0f\xfaS\xb1\xf9X\xa1\x11Y{\x14\x1c\x8e9"\x08=\xd7\x06aV\xea\xad\xf9\xfa\xfd7"\xff\xf8\xf4ze\xfd\x02\xbcPy3\xcb/l/\xe7B\x08\xc46\xf4\xc0\xc6\xdeq\xf2\x1c6\x8b\xe8qP\x87\xbbG\n\xc4\t\x81d\n\xea\x87\x17\xf2n\xf3i\x186\x17\xf3V7yv\xe2(5\x1d\tjlk U\xd6$\x8fP0G(\xa4\xd2T\xc8\xa0\xc2\xf1\'\xad\x11\'\xd8]\xcd\x93W\x0c\xadD9u{eS\x9b\n\x178.\xcf\x11\\._h.\x08\x0c\x03\x92\n\'\xba\x05\xda(\x91\x00\x9fDbx&\x8e\x8e\xbdk\x81\x94\x85\xa8\x08K\xdc\xb8\x94\'.\x11\x0c\x8d\xd1\xfb\x16\n\x00GS\x1b\xd7o\xe8\xb1"\xc5\x19\x9e\xcf\x9fM\x9d\x97/+\xeb\xa7\xdf\xb5\x1a\xa0(\x04\x9f\x17\xe2\xb9\x0e(iG}\xdc\xa6\xbc\xe5(\x0b\xe8c\xd8\x0f\x10\xbd\x03\xf9\xb4\xdf\xcfH\xae\x10\xbc\xb0\xbb)}\x876\x8d\xe2+\xa2\x15\xa6L`A\xb1\xa5~\xdf\xdc\xc2h+p`i\xda&Q\xc8\x8f\x96\x15Q0p8\xdd[?\xa5\xad\x84\x9d.\xad\xa6\xa0\xd4%y\xdb\x85pd\xa0\x8aR\x94\x89\xb3\x94\xb7\x8d@\xb0\xfb\xd1s\x19\xc7C\xc7\xd2\xef\xbe\x109X\xd2\x8a\xc5\xdbI\x16\xafU\x7f\x98\x111\x830\x0ba;\xe0\xa23\x04\x03\x0b\tt~\x14m\xe2\xa4([\xbe\x84\x04LqB\x8c\x1cgl\x9c\xc0v3j7\xb0\x08\xbfh\xd1u0\xdd|\x8b\xe7\xf6(\x03\x9b9\xe7\xd3\xe2^%\xb2\xe4\xa5\xcb_\x1d\xc8\xb2\xa4G\x1a\xa7\xf0,\x1b\xa6R\x8a:\x14\xf6~*\xa3\x9e\x8a\x92\xa8\xa2\x83"N\xba\xc9\xf1\x0f\x91\x0be\xd0Hv\x8dp\xeb{\xfa;\x01\x1b]z\x7fb\xd9\x8b\xe1g;)\r\x18\xba\xeb\x9c\x11"Z@q5-W;\x90\xa3\x91e`8\xb9\x80\t\x8e\x19E\x9ae\x05\xb60\x96d/-\xb3\xd4\xc3\xb3\x93\xbc\xa2\xde=\xe0\xefE+RNu\x13\x93\xbc\xdfR{N\x89~kb\xad\xbc\xb1\xc8\x02\xed\x94L\xa6\x06G\xf1\x8e!\x1e\xce\x105\xb8p/|\xc1\x9cH$\xe2 \xdc\x1a\x83\x7f\x04\'\xad;\xb9&zQ\xcf\x8bT\xb7_\xb6\xe8\x96b#\xb0\xf5-v~\xab\xa1\xd5\x19G\x86+\x19\xdbR\xb6a\x85\x96l]\xebo=\xd9\xb4\xe9s\x0f\xf8x\xab>\xba\xab\x89\xc6\xabg\x987d\xeaf+\xd2\xdf\x0e\x8c\x81}\xf7\x84\xa3\xa3W\xf7P\x17{\x95<\xb7\xdd\xf1{\x07rH\xf3\x9e=\x99\xb5K%N\xc6\x92\x06\x16\x9c\x80)K\xdc\x1b.\xb2\x86C\x11\xa1NCm\xe1\xfai\x88\r\xca\'\xbd\x8b\x07\x7f\xb1\xc8\x04\x8a@\xa9\xa8<;\xdeY\xd1f4?\xa4\xe33\xc6$\x83^\xbf\x7f\x80L4g\xbd\xf0\x1f\x8b\xca\x11\xf2b\x07n\x0f\xdc\x8ab\xb0,\xc3\xe7\xbb\xbb\x85\xe9\x94x\x8b-\x1d&\x187\\{\xa3;\xd0\x7f)\xe5\x884\xe0\xacy\xca\xce\x96\xd1A\xed\xc7\x0f}\x8d\x87\x88\xff1\xb5a\xffexR\xe9\xe8\xf2\xbac\x92\xa3\x98\xb7("i`\xbb\xdb\xe9\x11\x19\xfa\xf0\xc5\x8dCq59\xcdNw\xe1\xb0 \xf5\x10sT\xa8\xd1-_u\x8c\xb1\x87\xb3\xb3\x1cc\xa8]y\xd8\xb8~\xe7\x97\xa2+4\x87:j\xe7Jm|7\x83h\xd2\xaf\xeb\xfc\x8c\xd9\xb0\xb7$\xd2\xff`tlAt\xef\x02\x89\xf2\xbc\xf2\x94C\xa9\x1d\xcb \x9e\x04!2\xe9!\xe0\xd8&4\x9dFT+\xcbN\xdd\xb7\xb6\xd5\xe8\xa2\x16\xcd\x7f\x92s#F&>\xc4/U\xe7\x8a#\x9d\xd5! \xd0\xad\xe9r;+#\xed\x87\x97\xd0M\xf7\xb1\t\xa4\xc5\xfe1\x84\x1d+\xa8Zl\x8c5\xc9\xeb\xa5Ud\x8b\x88\xb2\xcd\xc8YC\xe4lA?\x8b\xc8\xbf\x14\xfeckk\x94U6\xafW/]\x15q.-\xc2\xe5\x89\xba5\x11\xf1\xe5 ]\xf5\xe2re\xe3\xf1?\x05Y9q\xddb\x07p\xb5\x9dC\xaa\x13i\xfdm\x1a2c\xff \xfez)CL\xd45\xb3\x7f\xcdL&\xcb\x9c\x95\xb55j\xb8\xe1Suy\x11%<\xf0){\x1bIk\xfe\xa7\xac?\xa8*\xfbb8\xadt\xe1\xb0\x9e\xdb\x16\xaa>\x0e\x96z\x83\x91\xe7\x85\\\x06\xccb*\x83\x07!\x18K\xe1\tjZ\xcf\x0e\xb0\x01\x1e.\xe8^k\xbe\xe5\x95\x9c\x87h\xfe}\xf5FO6R\x12\xbf\x95\x8f.\xe0Z\x90A\x85v#$\x80\xf7\x87nc\xd7\xeewY\xb7\x14?\xe2\xfb\xaf\x1d\x0c/&\xf5\xb5\xfe\xb5l\x15G\xbc\xf4\xb6T\x19C<\xdc\rUd\x88\xf8Fc\n\xaf\xac\x9d\x07K\x97C\x8d*l"\xa2\x90\x14\x94\xe4\x0e\xcb\x9c\x07\x87eKO\xdei\xf2\x86F\'\xc5\xda\xdfQ\xa5\x82\xa6\x9eXU&?\xe0\xa8\xe6\xef\xe2:\xe2\x14\xb6\xad\xa2\xd6\xe2\xf2\nzl\xab\xe04\xb9\x84\x85\x04\xa4\x8c\x19\xf7\xda-u\xb9\xbc\xfb\xfd%\xf7\xfc5u\xd1\xb0\xcc\x0b\x1d\xaf\xbf\xfd\x1a\xf5\x1a\ni\x19\x9f\xe5\x07\xb6a\xd4s\xfe\t]c\xbcM\xeb]\x05\xb4\xc5\x029\xed\x84\xd7\x89!A\x93\x9c\x8c\xe4\xd7K\xd9&hq \xb9\x92!Z\x87\nj\x03\xd4\xbcD&<\xd4\xcc7/\xd4\xd6/4\x16\xef\xc2R\xb6k\x96\x85$iD\xfc9G\x12_W\xff\x92\x8b\x18\x1f\x18\x9f\x1d:\x10\xc7\x95\'\xd0\xa5\x17\xa5\xf8\xa1\xd3\x1d\x01\xeb\x03\x1b\xe9Fb\xc7\xdbw}\xd4\xd1\xb1\xfb\xc4\xf6\\Q\xb6\xe3]\\\x10\x16,\rs;\xf9.\xc4\x96\xc9\x13\xe1\xe1\xd1-7\x91#\xfc\xe1\xddM\x04\x14\x1f\x8e*\x1f\x8d\x9e\xe8\xf8\xb1\xearB\xbfR\xef\xc4\xb6Q\x84%\xd3\x06\xf2\x98!\x88\xc4M\xd1\x02\xd3\x8b\x16\x87-\\\x08\x98\xcb\x94\r\xc0\xe9\x0c\xb4F\xd8pN\xe2f\xb7\xf86\xe8\x16W\xbdM\x82\x95\x17\xefp\xfe\x00\xdf\xbf\x90\x94\x0e\xa6|\x8eC\xc7^\x12X\xc6\xe4\x0c\x18\x95\xd5\xdb\xa4\x97,-s\x9d\x07\x87\x921\xb0\xc9\xed\xe4\x94=)\xef\xd4\x161\xac\xd0\xca\xb1V\xd0\nh\xba\x8d\xe3YNp@\x0b\x1fW\xcb*0\xda\x94e\x9fy%3 \xe0\xd9\xf1\x13\xd4= {\xbc\xe3\x84\xee\xc0\xca\xe7\x124j\xe4\xc4v\xc3\x08d\x81s\xfe\x12\xa0\xcb\x91\xbd\xc5E\xb6+\xf9b\xb0E\xa5\xad\xf5\x83i\xa1\xbd\xdd\x9c\xc7\xdbs\xb98F\xa6\x80\xb2&GYL\x17\xaaW\x9e\x85\xa5^m\xc7\tC\x88z_\'\\\xb9\x986\x05\xc1\xd2\x00\x9eN\xcam$`\x89\xa2/\xccV\x9e7\xd5\x10\x023\xbeiY\x9eU\xd9r\xb7\xb7\x82+Dz\xa1)\xc9k<\xde\xdey\xe2x\xe7\x89\x98\x97g\xb6\xadx\x9e\xd2\'XB+\xc9\xc4$$\xe9\x9d+A\xd8\xed\xe8\x19\x8fRz\x11\xe1\xb5\x08\xf3U6\xd6Q\xb0..368G\xc6s.\x9f?\x8f\x02\xf7\xd1\x1d\x8c\xe52\x8a\xae\x9e\x11\xfc\x7f\xe5\x05\xe6P\x8aD\x8c\xedm\xb8\xbc\xb0l\x0bF\xe0\'\t\xc0[$f\xfc7$\x7f"z\xbd\xa8]\xff\x9fW\xfa\xcc\xc2W\x92\xe7\xd5Q5\xc9\xafx\xc8_\xd1$c\xcd@mw\'\x04Z\xd9\x9d\xc3\x91T9\xb7Dt\x92\xeb\xe2\x9d-\xc4\x0e\xc7\x8cTq\x1c\tN!y\xc8:\xaf\x82 z>_\xbd~\x93sp\xb6\x8b\xd9~8\x04O\x86\xab\xa4\xc2\\\xa1\xfb\xb4Kt\xa2+w\x0c\x96\xc0QZ\x9eT\x06)\xe1\x85C\xdc\xa8\x0b\xa0\xc5I\x9e\x19-\xe8\x05rs\x9e\xba\xa8\xdc\xb9\xae2\xad\x85u\xecOw\x11\x14D\xb9;\xd5g\xdf\xd6\xceM\xb7\xd2F\x95\xea\x8dI\xc3i\xfb\x14\xee>\x9e~\x12\xaa\x0f\x8d\x8f;v\xc7;\xd3c\x06^/2\xff\xdd6u\xf1#\xfc\x8e\xb88B\x14\xa6s\x8b\x87v\xd2\x0f|\xa1<w\xfa%\xac\xc1\xbdxS\xad\x8a--|x\x07\xc4\xde\x0f\x8d\xa99\xec<0\xb1\n\xa7\xc1\x96+\x11\xb0E[\x80\xc5\xb9{%\xbc\xaaI\x9d\xbfB\xb5\xd4\x8ai\xbc\xb4\x16\xdb\x97\'\xef\xc1\xac\x05\xead\x928\xd6\x19-\xcf\xa2\x0eB\xc4\xe3)\xcf\x13\x17\xdd\xd5\xb3\xcd\xc7rhv\xcb62[\xfe\x8d:\xbd\xec=|\x8c\t\x86\x93r\xb4F\x98\xc5r\xce\x91>m\x96[A\xd5\xe9\x92\'w\xa8p\x80\xf3$\x10\xddD/\x957\x97\xd1\xdd\xab2Aa\xaaw\xe6\xb9`j\x94\x93\xb1\xacB\xb43\x16R|\xa0\x87\x1d\x84c\x08E\xa9\xed\x0c\xca\x1b\xee\x99\xa7\xa9EQ9\x93!U\x88\xf8\xd8\xeb\x05\xc9\xdc\x0b\xa7\x1d\x9e\x18Z\x0e\xb9\xd0!6/`\x89\xfd\xe3)\x95\xb7\xaa\x0cU/\x9d\xd3\x97\xf5/Mn\xcf\xb6\x0e\xd4\xac`k{\x86\x9c\xc5\xbd\xe9x\xe2]\x15\x1f\x0b\x87\'In\xbe\xa0}\x8b\xe9\x9a,\\~\xb7\x05Ki\xb7\xfc)V\xe2\xf3\xc7W\xa0!\xdcp"[\x8fM9-\x8d\x15\xc9\xf2\xea\x99\xda\xda\x1b/\xa98\x14\xae\x00\xfbG[e\xbb\xc5\xb6+\x11\xe4\x0c*j\xc7\xb0+\xf9G\xbe}Q\xd2#U\x02T\xdc\xb7(\xd4\tr8\x8d\x98\t\xe6<\xee\xe1>{\xf0\x96\xe5U\x1dk1\x90\xd4\x80%n\x00\xd1\x18]\xd2\x1f\xf0 \xa5iS%\xe5\x99\xdb\xc0\x1d\x8cL\xa62u\x12\xb3\x0b\x80\x13\xf1\x95\xcc\xc2\xeb@Z%~4\x15\xe5\x90\t@\xb19\xe91m\xce\xdcfXU\xe9\r\xaa\x02\xdc+D\xf1\xa6\x97\xacA(\x0cEXh\xf2?\xd7E\xf8\xe5\x1ei\xa5\xb6h$\xe2x\xf7Dj\xc2\xee\xb5\xd73W\xa4\xbc\xac|Uy}\x8f\x11\x81\x91\xb0\xf5\xf9\x7fF\xf1\x03\xd7\xc5\x88\xa2\xb7r\xbf\xe2\x98`\r\x04W\xc6\xe8a\xb9\xed\x97B\xe0\xf3\xcf\xf8\x1a.\x9cRHP\x90G|\xc5\x04HP\xa9(\xa8\xfb\x0f\xb9\xfd\xaeM\x15\x81\x9b7|\x9dv\xa7\xd6U3Z.\xc9\xf7\xd9\xa6\xe2\x06\x16\xd1\xed\x81\xae\xcd\xc2N\xf6\xdc\x16\x85B\x9f\xaa\xf7.O\x16\x89\xc5\x8b\x03\x99|\x7f4\xf2\x1bn|V\x08\xb4y/\xc4\x83\xad\x8a\xe4\x9d\xbf1Q\x8c@\x19\x90$[Q\xdc\x92\xee\xc1+\x137\xb3\xc1\x1a\xe6\x1e\x88\xde\xb4s\xab\xd5o\x1e\x01>\xd5\xa5\x17\xde\xa5j\xad \x16\x97\x7f!d[^FA\x97P\xc8[)b\xaa%\xaa\xbc\xb8\x94\x8cE\xa9\xb2\x89\x06\xc0y\x01(^\x90\xe7\x98C(m9\x89\xaf\xb9} \xfa\xe9\x1e\xdcu\xa7\xef\xfa\x04/N!\xd4]\x02\x93\xfa\xd9kG\x06\xbfYys\xb6\xba\xf4\x00wk\xfa\xb1\xbc\xadFW]y\'<\xddU]\x96\x1cp\xbb\x01\x86)\xf4\x81.)\xba\xc1\x16\x19Z\xa0\xdaF\xa5\x01\xf3_\xe6\x16\xf8ba&\x0b\xa6\x18.\xb3f\x97\x05j:>*\xe1V,\\\xa5.\x1c\xc3b-\'\xad\x91\xee\x8c\x99\xedO\x99t\x8f\xe0n\xfa\x17)\x01\xbbV8\xa5D\x16E\xb7\xac@3\x87\xbc\x8e\xab\xed\xd88\x82\xda\x1f2A\xc3F\xfd\xc9\x1c\xc0\xa2\x11\x0cnrx+_\xa4\x94\t\xcc\x95\x86\xb3\xe3\xc6\x12\'\xba\xd7\xa7\x9e\x06_\xf5\x1aK\xe3<\x87\x8e\x1d=|\xechb_\xef\x11\x80\xf5\xc8\xa1CG\x91\x94\xb3p\xc2\x03\xdc\x89\xc1\x9d\t\x81K\x81\xc4\x9e\x1ce|\xd3\xf5\x95~\xde\x1aQ\xc2\x07\xf4r \xce\x00k\xf9;\x14y\x93n\xab\xab\xb3\xaf\x96D\x0b\xf81\x8aE*\x1a\xb6\xeb\xc4p:w\xd8`\xf6\xd0\xc9.pW\xb0{n\xd5\xbbr\x0e\'\xe0\xf5$0\xe4\x00\x98\xd0\xd0B\xcd\xb4\xa9\xc3\xb9xxp\xcf\xe7=\xb8_Vv,\xd1\\"\x9e\x10%\xe2\xa3:\x1dh\xd7\n\xe2\\\x07\x84\xc8\xd2\xca\xdb\xc4u\x9b*i\x1e\xb32\x87\x8bv\x89\xcb\xf0l`^\xb1\x13\x80`\xe1Cz\xa0\x16\xc3+Q\x8ak\xfd\xfe\xd9\xfa\xa5\x19\xac\xc9;7\xeb\xa7M\xb1i\xdd\x86\xb6Z\x06\xe9 \xdf\xe8\xeby\x9c\xe5\xf5\x15\xd8\x0c\x04NX$\t\xf4f\xe2\xb1M-t\xf0\x86\xd1\xd3\xeb\xfa\xe5}\xfe\x0b\xb4\xd4%\x1a^\xb3\x84\xae6\xa3\x13\xac\xf8`\xd9-\x1d\xa8\xe4K\xc3\x0b\x8fUi\xabw\xf5\x06:W\xcb\xb9\x94\xd77\x05{\xb4\xaf\x06V\xaf\xfch\xa1=DJb\xc5q\xa6\xd2\x96\r\xa4|d&{\xf3\x88\xa2}\x81<b\xda@\x8fpy9\xf1*C\xc4\xc4\xbf\xf5\x95\x86\x9a\xed\xa9\xb7eQ\xc7Z\x0e\xb3-\xd5Mg\x12\x19\xb2\xb4\x03oG\xad\xac\xcd\xb0\x1b\x86\x11\xef\x9fN\x8e\x19\xb6\xe3q<v\xe8\xc8A\x85Q\xffB\x105\xed\xdb\xee\x8e\x9d\xa91\xff\xb7\x82\x88U?;\xd1\xf9\xde\xf3\x9d0G\x8a\xd8\x9e\xe8\x15\x1d\x14\x14$\xa3DI\x8f\xec}/\x03s\x93\ny~)\x15\xd2G\x9d\x19\xabM\x0c\xef/s\xe5\xc8\xa2S\xce\x94\xda|Fz\x1b\x87\x8aUuw\xd2Lb8\xd2\x1a4\x87l\xac\x19\x15\xea\xac\xb8\x14f\x82\x15o\xbc\xf0\x98r\xd3\xdd\xb7\x0f\xf14\xf0\xd7C\x89\x8e\x13|0\xd1\x9d*\x04\xcf\xd8\xcfo\x96\xf8\xe7\xc6{i\xe4A\xe9\x8f\x1d\x84\x9c\x92^%\xc7\xebR\xdd{\xa5nNy\xb7\x1fb\xebj\xca\x07MZ\x99\x0c\x1f\x06tl\xb1\x12,\xea\xd0\x94\xaa\xc0\xc5\xa0\xcf\xef\xeb=^9\xe9\x16\xab\xef\xd70\x19_\xae\x93.\xcb`\xf9\xa7h\xa9E\xc7Q\xa1\x859\x1c[\xf1\xae\xfa\x02JN\xa5\xb8\xc5\xbe\xb4r5\xe3\x97\x01\x16>\x1d\x16\xd8\xd8\xef\x1dI\x81J\xfd\x10rc/\xbcd\xc0\x8a\xea\xeb\x0fU\x951\x03\xec\xdf\x8f\xc8\xff^\xbad\xe0\x15\xb9\x9d\x9d\x98\x10\xa96G\r\xcc\xe3\xfd\xbcq\xad\xfepF\x04\x1c\xf0\xed.J\x9fl\xc2\x00\xec\xca\xff\x01-\xf4(\x91').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['t2v', 'i2v', 'l2v', 'flf2v', 'r2v']
REMOTE_MAX_IMAGES = 9

'''
Prepare remote-only defaults for tasks not selected manually in the Notebook.
遠端 API 為尚未手動執行的 Task 建立預設參數，不影響 Notebook 既有設定。
'''
TASK_CONFIGS = globals().setdefault('TASK_CONFIGS', {})
for task_name in REMOTE_ALLOWED_TASKS:
    if task_name not in TASK_CONFIGS:
        defaults = dict(prompt='A cinematic landscape with ambient sound.',
                        width=384, height=224, seconds=10, fps=FPS, steps=6 if REMOTE_SERVICE_ID.startswith('06_HyperFlow_') else 20,
                        seed=42, quant=QUANT, output_name=f'{task_name}.mp4',
                        first_image='', last_image='', ref_images=[], ref_videos=[], ref_audios=[],
                        video_audio=False, ref_image_size='match')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_'):
            defaults['audio_shift'] = 3.0 if task_name == 'r2v' else 6.0
        else:
            defaults['encoder_variant'] = ENCODER_VARIANT
        TASK_CONFIGS[task_name] = defaults

REMOTE_ROOT.mkdir(parents=True, exist_ok=True)
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=REMOTE_QUEUE_CAPACITY))
assert REMOTE_QUEUE.maxsize == REMOTE_QUEUE_CAPACITY, 'FIFO 容量已變更，請先重新啟動 Kernel 及遠端服務。'
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(REMOTE_UPLOAD_CONCURRENCY))
REMOTE_RESULT_TTL = REMOTE_RESULT_TTL_HOURS * 3600
REMOTE_RESULT_BYTES = REMOTE_RESULT_LIMIT_GIB * 1024**3
REMOTE_MIN_FREE_BYTES = REMOTE_MIN_FREE_GIB * 1024**3

'''
Do not start a second GPU job when a long-running backend appears stalled.
當底層生成疑似卡住時，停止接收新工作，絕不啟動第二筆 GPU 推論。
'''
REMOTE_STALL_SECONDS = REMOTE_STALL_HOURS * 3600  # 無限大代表不因等待時間長而拒絕工作

'''
Verify the CUDA backend and existing model without loading another model.
確認 CUDA 與已啟動的模型後端，不另外載入第二份模型。
'''
def remote_gpu_status():
    try:
        import torch
        if not torch.cuda.is_available():
            return False, 'CUDA GPU 不可用；請檢查 NVIDIA 驅動與 PyTorch，並重新執行模型初始化 Cell。', []
        qwen = REMOTE_KIND == 'image_multi'
        yue = REMOTE_SERVICE_ID.startswith('06_YuE2_')
        if yue:
            selected_gpus = [globals().get('GPU_ID', 0)]
        elif qwen and globals().get('QWEN21_STATE', {}).get('mode') != 'bf16_dual':
            selected_gpus = [globals().get('GPU_ID', 0)]
        else:
            selected_gpus = sorted({
                globals().get('MODEL_GPU_ID', 0),
                globals().get('TEXT_GPU_ID', 1) if REMOTE_SERVICE_ID.startswith(('06_MiniMax_', '06_HyperFlow_'))
                else globals().get('AUX_GPU_ID', 1) if REMOTE_SERVICE_ID.startswith(('06_HeartMuLa_', '06_LTX-2.5_', '06_Z-Image-Turbo_'))
                else globals().get('TEXT_GPU_ID', 1) if qwen else 0
            }) if not qwen else sorted({globals().get('GPU_ID', 0), globals().get('TEXT_GPU_ID', 1)})
        available = torch.cuda.device_count()
        if any(not isinstance(index, int) or index < 0 or index >= available for index in selected_gpus):
            return False, f'需要 cuda:{selected_gpus}，但實際只有 {available} 張 CUDA GPU。', []
        devices = [f'cuda:{index} {torch.cuda.get_device_name(index)}' for index in selected_gpus]
        if qwen:
            state = globals().get('QWEN21_STATE', {})
            if state.get('pipeline') is None or state.get('manager') is None:
                return False, 'Qwen Pipeline／GPU 記憶體管理器尚未初始化，請先執行模型初始化 Cell。', devices
            if state.get('device') != f"cuda:{globals().get('GPU_ID', 0)}":
                return False, 'Qwen Transformer 不在設定的 GPU 上。', devices
        elif REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
            pipeline = globals().get('pipe')
            if pipeline is None or pipeline.lazy_load or pipeline._mula is None or pipeline._codec is None:
                return False, 'HeartMuLa 雙 GPU Pipeline 尚未完整載入，請先執行模型載入 Cell。', devices
            for name, module, index in (('HeartMuLa', pipeline.mula, globals().get('MODEL_GPU_ID', 0)), ('HeartCodec', pipeline.codec, globals().get('AUX_GPU_ID', 1))):
                parameter = next(module.parameters(), None)
                if parameter is None or parameter.device != torch.device(f'cuda:{index}'):
                    return False, f'{name} 未位於指定的 cuda:{index}，請重新載入模型。', devices
        elif yue:
            gpu_id = globals().get('GPU_ID', 0)
            process = globals().get('SERVER_PROCS', {}).get(gpu_id)
            if process is None or process.poll() is not None:
                return False, f'YuE2 CUDA{gpu_id} 常駐服務尚未啟動。', devices
            if f'CUDA{gpu_id}' not in [str(value) for value in process.args]:
                return False, 'YuE2 常駐服務沒有使用預期的 CUDA 啟動參數。', devices
        else:
            '''
            Report CUDA as ready when this Notebook's ComfyUI is intentionally stopped.
            ComfyUI 已依設定停止時，允許下一筆工作重新啟動服務。
            '''
            process = globals().get('H3_PROC')
            if (globals().get('VIDEO_RESTART_AFTER_GENERATION', False)
                    and REMOTE_SERVICE_ID.startswith(('06_MiniMax_H3_', '06_HyperFlow_H3_'))
                    and process is not None and process.poll() is not None):
                return True, 'CUDA 可用；ComfyUI 將於下一筆工作開始前啟動。', devices
            base = globals().get('BASE')
            if not base and REMOTE_IS_LTX:
                base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
            if not isinstance(base, str) or not base:
                return False, 'ComfyUI 尚未啟動，請先執行模型服務 Cell。', devices
            with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=4) as response:
                stats = json.load(response)
            comfy_devices = stats.get('devices', [])
            if not any(str(item.get('type', '')).lower().startswith('cuda') or
                       str(item.get('name', '')).lower().startswith('cuda:')
                       for item in comfy_devices if isinstance(item, dict)):
                return False, 'ComfyUI 沒有回報 CUDA 裝置，已拒絕 CPU-only 推論。', devices
        return True, 'CUDA 後端已就緒（不代表所有權重常駐 GPU 記憶體）。', devices
    except Exception as error:
        return False, f'GPU 檢查失敗：{type(error).__name__}: {error}', []

'''
Verify every global required by the selected Notebook generation engine.
驗證所選模型的生成程式所需變數與設定已完成初始化。
'''
def remote_inference_ready(task):
    namespace = globals()
    common = {
        '06_HeartMuLa_': ('ROOT', 'pipe'),
        '06_YuE2_': ('ROOT', 'GPU_ID', 'PORTS', 'SERVER_PROCS'),
        '06_Z-Image-Turbo_': ('ROOT', 'BASE', 'LOG_PATH'),
        '06_Qwen_Image_': ('MODEL_MODE', 'GPU0_MODEL_GIB', 'GPU_ID', 'QWEN21_STATE'),
        '06_LTX-2.5_': ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG', 'COMFY_PROC',
                          'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB', 'importlib'),
        '06_MiniMax_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                              'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                              'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                              'QUANT_SIZES_GIB', 'ready', 'choices', 'check_choice'),
        '06_HyperFlow_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                                'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                                'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                                'QUANT_SIZES_GB', 'ready', 'choices', 'ENCODERS',
                                'ENCODER_FILE', 'MODEL_FAMILY', 'WEIGHTS_IN',
                                'TEXT_LOADER_NODE', 'PROJECTION_FILE'),
    }
    required = next((names for prefix, names in common.items()
                     if REMOTE_SERVICE_ID.startswith(prefix)), ())
    missing = [name for name in required if name not in namespace or namespace[name] is None]
    if missing:
        return False, 'Notebook 尚未完成初始化，請先執行對應的模型／服務 Cells；缺少：' + ', '.join(missing)
    if task not in REMOTE_ALLOWED_TASKS:
        return False, '此模型不支援生成模式：' + task
    if REMOTE_SERVICE_ID.startswith(('06_MiniMax_H3_', '06_HyperFlow_H3_')):
        config = namespace['TASK_CONFIGS'].get(task)
        if not isinstance(config, dict):
            return False, f'{task.upper()} 尚未設定；請先執行 Notebook 對應的模式設定 Cell。'
        if namespace['H3_ACTIVE_CONFIG'] != namespace['H3_DESIRED_CONFIG']:
            return False, 'GPU／Offload 設定與已啟動 ComfyUI 不一致，請重新啟動模型服務 Cell。'
        quant = config.get('quant')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            variant = 'ref2va_pruned' if task == 'r2v' else 'fl2va_pruned'
            expected = (variant, quant)
            expected_file = f'minimax_h3_{variant}-{quant}.gguf'
        else:
            variant = 'ref2va' if task == 'r2v' else 'fl2va'
            encoder = config.get('encoder_variant')
            expected = (variant, quant, encoder)
            expected_file = f'minimax_h3_{variant}_hyperflow_flash_14b-{quant}.gguf'
            if encoder not in namespace['ENCODERS']:
                return False, f'{task.upper()} 的 Encoder 設定無效，請重新執行模式設定 Cell。'
            if Path(namespace['ENCODER_FILE']).name != namespace['ENCODERS'][encoder][1]:
                return False, 'GPU 1 Encoder 與此模式不一致，請重新執行模型下載／啟動 Cells。'
        if namespace['H3_MODEL_CHOICE'] != expected or namespace['H3_ACTIVE_MODEL'] != expected:
            return False, f'{task.upper()} 尚未載入相應模型權重；請設定模式並重新執行模型下載／啟動 Cells。'
        if namespace['DIT_NAME'] != expected_file:
            return False, 'ComfyUI 使用的模型檔名與任務不一致，請重新載入正確模型。'
    elif REMOTE_SERVICE_ID.startswith('06_Qwen_Image_'):
        state = namespace['QWEN21_STATE']
        if state.get('mode') != namespace['MODEL_MODE']:
            return False, 'Qwen MODEL_MODE 與已初始化的模型模式不同，請重新執行模型初始化 Cell。'
        if state.get('gpu0_model_gib') != namespace['GPU0_MODEL_GIB']:
            return False, 'Qwen GPU0_MODEL_GIB 與目前載入設定不同，請重新初始化 GPU 記憶體管理器。'
    elif REMOTE_IS_LTX:
        if not Path(namespace['ENGINE_PATH']).is_file():
            return False, 'LTX 模型引擎檔案不存在；請先執行模型與 Workflow Cell。'
        if namespace['COMFY_PROC'].poll() is not None:
            return False, 'LTX ComfyUI 行程已停止，請重新執行啟動 ComfyUI Cell。'
    return True, '模型初始化與所選模式已就緒。'

'''
Validate client parameters before they occupy a FIFO slot.
在進入 FIFO 前檢查 client 參數，避免排隊後才因設定錯誤而失敗。
'''
def remote_validate_fields(task, fields):
    required = {'task'}
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        required.update(('lyrics', 'tags', 'duration', 'seed'))
    elif REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        required.update(('style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot'))
    elif REMOTE_KIND == 'image_text':
        required.update(('prompt', 'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_KIND == 'image_multi':
        required.update(('prompt', 'negative_prompt', 'sampler', 'reference_size',
                         'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_IS_LTX:
        required.update(('prompt', 'width', 'height', 'seconds', 'fps', 'seed', 'image_strength'))
    else:
        required.update(('prompt', 'width', 'height', 'seconds', 'fps', 'steps', 'seed', 'ref_image_size'))
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            required.add('audio_shift')
    missing = required - fields.keys()
    if missing:
        raise ValueError('請求缺少必要參數：' + ', '.join(sorted(missing)))
    integer_fields = ('seed', 'lm_seed', 'width', 'height', 'steps', 'fps', 'duration', 'reference_size')
    float_fields = ('seconds', 'cfg', 'image_strength', 'audio_shift')
    numbers = {}
    for key in integer_fields:
        if key in fields:
            raw = fields[key]
            if isinstance(raw, bool) or str(raw).strip() != str(int(raw)):
                raise ValueError(f'{key} 必須是整數')
            numbers[key] = int(raw)
    for key in float_fields:
        if key in fields:
            numbers[key] = float(fields[key])
            if not math.isfinite(numbers[key]):
                raise ValueError(f'{key} 必須是有限數值')
    for key in ('seed', 'lm_seed'):
        if key in numbers and not -1 <= numbers[key] < 2**32:
            raise ValueError(f'{key} 必須介於 -1 至 4294967295')
    for key in ('width', 'height'):
        if key in numbers:
            size = numbers[key]
            divisor = 64 if REMOTE_KIND == 'image_text' else 32
            if size < 1 or size > 4096 or size % divisor:
                raise ValueError(f'{key} 必須介於 1～4096 且為 {divisor} 的倍數')
    for key in ('seconds', 'duration'):
        if key in numbers and not 0 < numbers[key] <= 3600:
            raise ValueError(f'{key} 必須大於 0 且不超過 3600')
    '''
    Reject unsupported FPS values before placing jobs in the GPU queue.
    在工作排入 GPU 佇列前，先拒絕不支援的 FPS 設定。
    '''
    if 'fps' in numbers:
        if REMOTE_IS_LTX and not 1 <= numbers['fps'] <= 60:
            raise ValueError('LTX-2.5 FPS 必須介於 1～60')
        if not REMOTE_IS_LTX and numbers['fps'] != 24:
            raise ValueError('MiniMax H3／HyperFlow H3 僅支援 24 FPS')
    if 'steps' in numbers and not 1 <= numbers['steps'] <= 200:
        raise ValueError('steps 必須介於 1～200')
    if REMOTE_SERVICE_ID.startswith('06_HyperFlow_H3_') and numbers.get('steps') not in (4, 6, 8):
        raise ValueError('HyperFlow steps 只支援 4、6、8')
    if 'image_strength' in numbers and not 0 <= numbers['image_strength'] <= 1:
        raise ValueError('image_strength 必須介於 0～1')
    if 'reference_size' in numbers and not 64 <= numbers['reference_size'] <= 2048:
        raise ValueError('reference_size 必須介於 64～2048')
    if 'audio_shift' in numbers and not 0.01 <= numbers['audio_shift'] <= 100:
        raise ValueError('audio_shift 必須介於 0.01～100')
    if fields.get('ref_image_size', 'match') not in ('match', 'max'):
        raise ValueError('ref_image_size 只支援 match 或 max')
    if REMOTE_KIND in ('video', 'image_text', 'image_multi') and not str(fields.get('prompt', '')).strip():
        raise ValueError('PROMPT 不可為空白')
    if task == 'r2v':
        image_count = int(fields.get('_image_count', 0))
        video_count = int(fields.get('_video_count', 0))
        audio_count = int(fields.get('_audio_count', 0))
        if not (image_count or video_count):
            raise ValueError('R2V 至少需要參考圖片或影片，不能只提供音訊。')
        if image_count > 9 or video_count > 3 or audio_count > 3 or image_count + video_count + audio_count > 12:
            raise ValueError('R2V 最多 9 張圖片、3 段影片、3 段音訊，合計最多 12 個素材。')
        count_by_type = {'Picture': image_count, 'Video': video_count,
                         'Audio': audio_count + (video_count if str(fields.get('video_audio', 'false')).lower() == 'true' else 0)}
        for media, count in count_by_type.items():
            matches = [int(number) for number in re.findall(r'<%s\s+(\d+)>' % media, str(fields['prompt']), re.I)]
            if any(number < 1 or number > count for number in matches):
                raise ValueError(f'R2V Prompt 的 <{media} N> 編號超出已上傳的參考素材數量。')

'''
Report whether the model is ready for CUDA inference.
回報模型是否已具備 CUDA 推論條件。
'''
def remote_backend_ready():
    return remote_gpu_status()[0]

'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            gpu_ready, gpu_message, gpu_devices = remote_gpu_status()
            requested_task = parse_qs(urlsplit(self.path).query).get('task', [REMOTE_ALLOWED_TASKS[0]])[-1]
            inference_ready, inference_message = remote_inference_ready(requested_task)
            if gpu_ready and not inference_ready:
                gpu_message = inference_message
            self.send_json(200, {'ok': True, 'api': 'kaggle-generation-bridge-v3',
                                 'service_id': REMOTE_SERVICE_ID, 'task': requested_task,
                                 'backend_ready': gpu_ready and inference_ready, 'gpu_ready': gpu_ready,
                                 'gpu_message': gpu_message, 'gpu_devices': gpu_devices,
                                 'stalled': remote_stalled(), 'running': running,
                                 'waiting': REMOTE_QUEUE.qsize(), 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES, 'video_limit': 3, 'audio_limit': 3,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        gpu_ready, gpu_message, _ = remote_gpu_status()
        if not gpu_ready:
            self.send_json(409, {'error': gpu_message})
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Kaggle disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files, videos, audios = {}, [], [], []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name in ('images', 'videos', 'audios'):
                        collection = {'images': files, 'videos': videos, 'audios': audios}[name]
                        collection.append((part.get_filename() or 'upload', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES or len(videos) > 3 or len(audios) > 3:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and (files or videos or audios):
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'l2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (0, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'l2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if task != 'r2v' and (videos or audios):
                raise ValueError('Reference videos/audio are supported only for R2V')
            if task == 'r2v' and not (files or videos):
                raise ValueError('R2V requires an image or a video; audio alone is not supported')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)
            remote_validate_fields(task, dict(fields, _image_count=len(files), _video_count=len(videos), _audio_count=len(audios)))
            if 'quant' in fields and fields['quant']:
                active_model = globals().get('H3_ACTIVE_MODEL', ())
                if fields['quant'] not in active_model:
                    self.send_json(409, {'error': '指定的量化尚未載入；請先在 Notebook 下載並預檢該 GGUF。'})
                    return
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                self.send_json(409, {'error': inference_message})
                return
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            if str(fields.get('video_audio', 'false')).lower() not in ('true', 'false'):
                raise ValueError('video_audio must be true or false')
            for filename, data in videos:
                if not data or len(data) > 80 * 1024 * 1024 or Path(filename).suffix.lower() not in ('.mp4', '.mov', '.webm'):
                    raise ValueError('Reference video must be MP4/MOV/WebM and at most 80 MiB')
            for filename, data in audios:
                if not data or len(data) > 30 * 1024 * 1024 or Path(filename).suffix.lower() not in ('.mp3', '.wav', '.flac', '.ogg', '.m4a'):
                    raise ValueError('Reference audio must be an audio file at most 30 MiB')
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names, video_names, audio_names = [], [], []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)
            for kind, uploaded, names in (('video', videos, video_names), ('audio', audios, audio_names)):
                for index, (filename, data) in enumerate(uploaded, 1):
                    target = staging / f'{kind}_{index:03d}{Path(filename).suffix.lower()}'
                    target.write_bytes(data)
                    names.append(target.name)

            '''
            Commit the validated upload and queue entry in one critical section.
            在同一個鎖定區段確認請求、保存資料並加入工作佇列。
            '''
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    fields['_ref_videos'] = [str(folder / name) for name in video_names]
                    fields['_ref_audios'] = [str(folder / name) for name in audio_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED',
                'QUANT', 'ENCODER_VARIANT']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'fps': int, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value
            for seed_key in ('seed', 'lm_seed'):
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'fps', 'ref_image_size', 'quant'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    updated['quant'] = inputs.get('quant') or namespace.get('QUANT', updated.get('quant'))
                    if task == 'r2v':
                        updated['video_audio'] = str(inputs.get('video_audio', 'false')).lower() == 'true'
                        updated['ref_videos'] = inputs.get('_ref_videos', [])
                        updated['ref_audios'] = inputs.get('_ref_audios', [])
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'l2v':
                        updated['last_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        namespace['R2V_REF_DIR'] = REMOTE_ROOT / job_id
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)
            '''
            Recheck the CUDA backend after queuing to prevent CPU-only generation.
            工作排入佇列後再次確認 CUDA，避免後端異常時改走純 CPU 生成。
            '''
            '''
            Start the local service before GPU and model checks.
            GPU 與模型檢查前，先啟動本機 ComfyUI 服務。
            '''
            if VIDEO_RESTART_AFTER_GENERATION:
                h3_ensure_service()
            gpu_ready, gpu_message, _ = remote_gpu_status()
            if not gpu_ready:
                raise RuntimeError(gpu_message)
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                raise RuntimeError(inference_message)
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            '''
            Stop ComfyUI only after the API has saved the generated result.
            確認遠端 API 已儲存影片後，才停止 ComfyUI 回收 RAM。
            '''
            if VIDEO_RESTART_AFTER_GENERATION:
                h3_stop_service()
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()

if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)

NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(ROOT / 'ngrok')
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')

result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')

del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()

if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')

with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)

NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)

if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])

print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')
print('✓ 第 14 節完成：選用 FIFO／ngrok API 已啟動。')


15. （選用）遠端 API 與 client 使用說明


In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)

print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本機 requests 範例：06_MiniMax_H3_影片生成_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')
print('✓ 第 15 節完成：API Schema 和 client 說明已顯示。')
